Code initiation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as dsets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import make_axes_locatable
from torch.utils.data import DataLoader
from math import floor 
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import mutual_info_score

# Check if CUDA is available
device1 = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("mps" if torch.backends.mps.is_available() else device1)
print(f"Using device: {device}")

Using device: cuda


Define Hyperparameters and Data Loading

In [2]:
# Define Hyperparameters
input_size = 784 * 2  # img_size = (28,28) ---> 28*28=784 in total
num_classes = 10  # number of output classes, discrete range [0,9]
num_epochs = 200  # number of times which the entire dataset is passed through the model
lr = 1e-2  # learning rate
scheduler_gamma = 0.98

# Define Dendrite Hyperparameters
excitatory_synapses_per_dendrite_branch = 50
inhibitory_synapses_per_dendrite_branch = 1
dendrite_layers_n = 5
dendrite_branch_per_dendrite = [4, 4, 4, 4, 4]
dendrite_per_soma = 10
e_to_i_ratio = excitatory_synapses_per_dendrite_branch / inhibitory_synapses_per_dendrite_branch
soma_n = num_classes
synapses_per_inhibitory_unit = 200
inhibitory_units_n = 100
sigmoid_offset = 5
dendrite_branch_n = [None] * dendrite_layers_n
dendrite_branch_n[0] = dendrite_per_soma * soma_n
shunting = True

for i in range(1, dendrite_layers_n):
    dendrite_branch_n[i] = dendrite_branch_n[i - 1] * dendrite_branch_per_dendrite[i - 1]

print(dendrite_branch_n)
sum(dendrite_branch_n)

# Downloading MNIST data
train_data = dsets.MNIST(root='./data', train=True, transform=transforms.ToTensor(), download=True)
test_data = dsets.MNIST(root='./data', train=False, transform=transforms.ToTensor())

# Loading the data
batch_size = 1000
train_gen = torch.utils.data.DataLoader(dataset=train_data, batch_size=batch_size, shuffle=True)
test_gen = torch.utils.data.DataLoader(dataset=test_data, batch_size=batch_size, shuffle=False)

# Index arrays
index_list1 = torch.tensor(range(batch_size))
index_list2 = torch.tensor(list(range(1, batch_size)) + [0])
modulo_base = num_classes

def sum_labels(labels_in):
    return torch.remainder(labels_in[index_list1] + labels_in[index_list2], modulo_base)

def transform_images(images_in):
    return torch.cat((images_in[index_list1, :], images_in[index_list2, :]), 1)


[100, 500, 2500, 12500, 62500]


Utilities

In [3]:
import numpy as np
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import mutual_info_score

def add_noise_to_features(features, noise_level=1e-6):
    """
    Add small noise to each feature to avoid constant features.
    
    Parameters:
    features (numpy array): The input feature matrix.
    noise_level (float): The standard deviation of the Gaussian noise to be added.
    
    Returns:
    numpy array: The feature matrix with added noise.
    """
    noise = np.random.normal(0, noise_level, features.shape)
    return features + noise

def compute_mutual_information(activations, labels, n_bins=10, noise_level=1e-6):
    # Ensure labels is 1D
    if labels.ndim == 2 and labels.shape[1] == 1:
        labels = labels.ravel()  # Convert to 1D array

    mi = []
    discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='uniform')
    
    # Reshape to 2D array with a single feature
    b = activations.reshape(-1, activations.shape[-1])
    b = np.array(b)
    
    # Add noise to features
    b_noisy = add_noise_to_features(b, noise_level)

    for i in range(b_noisy.shape[1]):
        a = b_noisy[:, i].reshape(-1, 1)
        
        activations_discretized = discretizer.fit_transform(a)
        
        binary_labels = (labels == i).astype(int)  # Create binary labels for the current unique label
        mi.append(mutual_info_score(binary_labels, activations_discretized[:, 0]))

    return np.array(mi)


def compute_mutual_information_by_epoch(activations_by_epoch, labels_raw, num_epochs, num_dendrites):
    mi_by_epoch = {i: [] for i in range(num_epochs)}

    labels_raw = labels_raw.unsqueeze(1) if labels_raw.dim() == 1 else labels_raw

    labels1 = labels_raw[:, 0].cpu().numpy()
    labels2 = labels_raw[:, 1].cpu().numpy()
    labels_sum_mod = sum_labels(labels_raw).cpu().numpy()

    for epoch in range(num_epochs):
        activations = activations_by_epoch[epoch]
        for d in range(num_dendrites):
            mi_d1 = mutual_info_regression(activations[:, d].reshape(-1, 1), labels1)
            mi_d2 = mutual_info_regression(activations[:, d].reshape(-1, 1), labels2)
            mi_sum = mutual_info_regression(activations[:, d].reshape(-1, 1), labels_sum_mod)
            mi_by_epoch[epoch].append((mi_d1, mi_d2, mi_sum))

    return mi_by_epoch


def analyze_weights(net):
    weights = {}
    for name, param in net.named_parameters():
        if 'weight' in name:
            weights[name] = param.data.cpu().numpy()
    
    for name, weight in weights.items():
        plt.figure(figsize=(8, 6))
        plt.hist(weight.flatten(), bins=50)
        plt.title(f'Weight Distribution: {name}')
        plt.show()
    
    return weights

Utility Functions and Classes

In [4]:
# Function to plot an image with the scale color bar
def plotWithColor(data, title=''):
    if torch.is_tensor(data):
        data = data.cpu().numpy()
    ax = plt.subplot()
    im = ax.imshow(data)
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax)
    if len(title) > 0:
        plt.title(title)
    plt.show()


# Layer class that only keeps K top weights %original
class TopKLinear(nn.Linear):
    """ Custom Linear layer that only uses the strongest K weights """
    def __init__(self, in_features, out_features, keep_in_features, bias=False):
        super().__init__(in_features, out_features, bias=bias)
        self.keep_in_features = keep_in_features
        self.topk_indices = []
        self.mask = torch.zeros_like(self.weight)
        self.net_weight = torch.zeros_like(self.weight)
        self.inhibition_act = None  # Initialize inhibition_act attribute

    def forward(self, x):
        self.mask = self.mask.to(self.weight.device)
        self.topk_indices = torch.topk(self.weight, self.keep_in_features, 1, largest=True)[1]
        self.mask.scatter_(1, self.topk_indices.to(self.mask.device), 1)
        self.net_weight = self.mask * self.weight
        w_times_x = torch.mm(x, self.net_weight.t())
        self.inhibition_act = w_times_x  # Store activation values for analysis
        return w_times_x

# Layer class that uses a block structure
class BlockLinear(nn.Linear):
    """ Custom Linear layer that uses block structures """
    def __init__(self, in_features, out_features, bias=False):
        super().__init__(in_features, out_features, bias=bias)
        block_size = floor(in_features / out_features)
        assert(block_size * out_features == in_features)
        self.block_size = block_size
        self.mask = torch.zeros(out_features, in_features, device=device)
        for counter in range(self.out_features):
            self.mask[counter, torch.arange(counter * self.block_size, (1 + counter) * self.block_size)] = 1

    def forward(self, x):
        self.net_weight = self.weight * self.mask
        w_times_x = torch.mm(x, self.net_weight.t())
        return w_times_x

class DendriteBranchLayer(nn.Module):
    def __init__(self, input_size, inhibitory_units_n, dendrite_branch_n, inhibitory_synapses_per_dendrite_branch, 
                 excitatory_synapses_per_dendrite_branch, shunting=False, branch_input=False, input_dendrite_branch_n=None, e_to_i_ratio=1):
        super(DendriteBranchLayer, self).__init__()

        self.branch_input = branch_input
        self.shunting = shunting

        #self.inputs_to_dendrite_branch = nn.Linear(input_size, dendrite_branch_n)
        #self.inhibition_to_dendrite_branch = nn.Linear(inhibitory_units_n, dendrite_branch_n)
            
        self.inputs_to_dendrite_branch = TopKLinear(input_size, dendrite_branch_n, excitatory_synapses_per_dendrite_branch)
        self.inhibition_to_dendrite_branch = TopKLinear(inhibitory_units_n, dendrite_branch_n, inhibitory_synapses_per_dendrite_branch)

        self.coefficients = nn.Parameter(torch.ones(dendrite_branch_n))

        if self.branch_input:
            #self.dendrite_branch_inputs_to_dendrite_branch = nn.Linear(input_dendrite_branch_n, dendrite_branch_n)
            self.dendrite_branch_inputs_to_dendrite_branch = BlockLinear(input_dendrite_branch_n, dendrite_branch_n)
            self.dendrite_branch_inputs_to_dendrite_branch.requires_grad_(False)  # We don't train this. All dendrites have the same weight
        else:
            self.dendrite_branch_inputs_to_dendrite_branch = None

        self.sigmoid = nn.Sigmoid()
        self.e_to_i_ratio = e_to_i_ratio

    def forward(self, input, inhibition_outputs, dendrite_branch_outputs=None):
        self.dendrite_inhibition = self.inhibition_to_dendrite_branch(inhibition_outputs)
        #excitation_outputs
        self.dendrite_excitation = self.inputs_to_dendrite_branch(input)

        if self.branch_input:
            self.dendrite_depolarization = self.dendrite_branch_inputs_to_dendrite_branch(dendrite_branch_outputs)
            total_excitatory_input = self.dendrite_excitation + self.dendrite_depolarization
        else:
            total_excitatory_input = self.dendrite_excitation

        if self.shunting:
            self.dendrite_act = total_excitatory_input / \
            (total_excitatory_input + self.e_to_i_ratio * self.dendrite_inhibition + 1)
            #self.dendrite_act = self.sigmoid(self.dendrite_act)
        else:
            self.dendrite_act = total_excitatory_input - self.e_to_i_ratio * self.dendrite_inhibition - 3
            self.dendrite_act = self.sigmoid(self.dendrite_act)

            self.dendrite_act = self.coefficients * self.dendrite_act

        return self.dendrite_act, self.dendrite_inhibition, self.dendrite_excitation
    
    
    
    

Define the Network Classes

In [5]:
class Net(nn.Module):
    def __init__(self, input_size, inhibitory_units_n, synapses_per_inhibitory_unit, dendrite_layers_n, dendrite_branch_n, 
                 inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, shunting, soma_n):
        super(Net, self).__init__()

        self.shunting = shunting
        self.dendrite_layers_n = dendrite_layers_n

        self.inputs_to_inhibition = TopKLinear(input_size, inhibitory_units_n, synapses_per_inhibitory_unit)

        self.dendrite_layers = nn.ModuleList([None] * dendrite_layers_n)
        
        for i in range(dendrite_layers_n - 1, -1, -1):
            is_outermost_layer = (i == (dendrite_layers_n - 1))
            branch_input = False if is_outermost_layer else True
            input_dendrite_branch_n = None if is_outermost_layer else dendrite_branch_n[i + 1]
            self.dendrite_layers[i] = DendriteBranchLayer(input_size, inhibitory_units_n, dendrite_branch_n[i], 
                                                          inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, 
                                                          shunting, branch_input, input_dendrite_branch_n)

        self.dendrite_to_soma = BlockLinear(dendrite_branch_n[0], soma_n)
        self.dendrite_to_soma.requires_grad_(False)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

        # Initialize containers for storing activations and inputs
        self.dendrite_activations = [None] * dendrite_layers_n
        self.inhibitory_inputs = [None] * dendrite_layers_n
        self.excitatory_inputs = [None] * dendrite_layers_n

    def initWeights(self):
        with torch.no_grad():
            # Initialize weights using a uniform distribution to avoid gradient issues
            nn.init.kaiming_uniform_(self.inputs_to_inhibition.weight, a=0, mode='fan_in', nonlinearity='relu')
            for i in range(self.dendrite_layers_n):
                nn.init.kaiming_uniform_(self.dendrite_layers[i].inputs_to_dendrite_branch.weight, a=0, mode='fan_in', nonlinearity='relu')
                nn.init.kaiming_uniform_(self.dendrite_layers[i].inhibition_to_dendrite_branch.weight, a=0, mode='fan_in', nonlinearity='relu')
            nn.init.kaiming_uniform_(self.dendrite_to_soma.weight, a=0, mode='fan_in', nonlinearity='relu')

    def trimWeights(self):
        with torch.no_grad():
            # Clamp weights to non-negative values to ensure proper training behavior
            self.inputs_to_inhibition.weight.data = torch.clamp(self.inputs_to_inhibition.weight, min=0)
            for i in range(self.dendrite_layers_n):
                self.dendrite_layers[i].inputs_to_dendrite_branch.weight.data = torch.clamp(self.dendrite_layers[i].inputs_to_dendrite_branch.weight, min=0)
                self.dendrite_layers[i].inhibition_to_dendrite_branch.weight.data = torch.clamp(self.dendrite_layers[i].inhibition_to_dendrite_branch.weight, min=0)
            self.dendrite_to_soma.weight.data = torch.clamp(self.dendrite_to_soma.weight, min=0)

    def forward(self, input, labels):
        
        #input = input.view(input.size(0), -1)  # Flatten the input tensor
        
        self.inhibition_act = self.inputs_to_inhibition(input)
        if self.shunting:
            self.inhibition_out = self.inhibition_act / (1 + self.inhibition_act)
        else:
            self.inhibition_out = self.sigmoid(self.inhibition_act)
        
        self.dendrite_branch_out = [None] * self.dendrite_layers_n

        for i in range(self.dendrite_layers_n - 1, -1, -1):
            is_outermost_layer = (i == (self.dendrite_layers_n - 1))
            this_dendrite_branch_out = None if is_outermost_layer else self.dendrite_branch_out[i + 1]
            self.dendrite_branch_out[i], self.inhibitory_inputs[i], self.excitatory_inputs[i] = self.dendrite_layers[i](input, self.inhibition_out, this_dendrite_branch_out)
            self.dendrite_activations[i] = self.dendrite_branch_out[i]

        self.soma_act = self.dendrite_to_soma(self.dendrite_branch_out[0])
        return self.soma_act, labels
    
    def l1_loss(self):
        l1_loss = 0
        for layer in self.dendrite_layers:
            l1_loss += torch.norm(layer.coefficients, 1)
        return self.lambda_l1 * l1_loss
    
class NetWithAttention(nn.Module):
    def __init__(self, input_size, inhibitory_units_n, synapses_per_inhibitory_unit, dendrite_layers_n, dendrite_branch_n, 
                 inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, shunting, soma_n, lambda_l1=0.001):
        super(NetWithAttention, self).__init__()

        self.shunting = shunting
        self.dendrite_layers_n = dendrite_layers_n
        self.lambda_l1 = lambda_l1  # L1 regularization strength

        # Inhibitory cell layer
        self.inputs_to_inhibition = TopKLinear(input_size , inhibitory_units_n, synapses_per_inhibitory_unit)

        # Dendrite layers
        self.dendrite_layers = nn.ModuleList([None] * dendrite_layers_n)
        for i in range(dendrite_layers_n - 1, -1, -1):
            is_outermost_layer = (i == (dendrite_layers_n - 1))
            branch_input = False if is_outermost_layer else True
            input_dendrite_branch_n = None if is_outermost_layer else dendrite_branch_n[i + 1]
            self.dendrite_layers[i] = DendriteBranchLayer(input_size , inhibitory_units_n, dendrite_branch_n[i], 
                                                          inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, 
                                                          shunting, branch_input, input_dendrite_branch_n)

        # Attention layers
        self.attention_layers = nn.ModuleList([nn.Linear(dendrite_branch_n[i], dendrite_branch_n[i]) for i in range(dendrite_layers_n)])

        # Soma
        self.dendrite_to_soma = BlockLinear(dendrite_branch_n[0], soma_n)
        self.dendrite_to_soma.requires_grad_(False)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

        # Initialize containers for storing activations and inputs
        self.dendrite_activations = [None] * dendrite_layers_n
        self.inhibitory_inputs = [None] * dendrite_layers_n
        self.excitatory_inputs = [None] * dendrite_layers_n

    def initWeights(self):
        with torch.no_grad():
            self.inputs_to_inhibition.weight.data = torch.sign(self.inputs_to_inhibition.weight) * self.inputs_to_inhibition.weight
            for i in range(self.dendrite_layers_n - 1, -1, -1):
                self.dendrite_layers[i].inputs_to_dendrite_branch.weight.data = torch.sign(self.dendrite_layers[i].inputs_to_dendrite_branch.weight) * self.dendrite_layers[i].inputs_to_dendrite_branch.weight
                self.dendrite_layers[i].inhibition_to_dendrite_branch.weight.data = torch.sign(self.dendrite_layers[i].inhibition_to_dendrite_branch.weight) * self.dendrite_layers[i].inhibition_to_dendrite_branch.weight
                if i < (self.dendrite_layers_n - 1):
                    self.dendrite_layers[i].dendrite_branch_inputs_to_dendrite_branch.weight.data = 1.0 * self.dendrite_layers[i].dendrite_branch_inputs_to_dendrite_branch.mask

            self.dendrite_to_soma.weight.data = 1.0 * self.dendrite_to_soma.mask

    def trimWeights(self):
        with torch.no_grad():
            self.inputs_to_inhibition.weight.data = torch.maximum(self.inputs_to_inhibition.weight, 0 * self.inputs_to_inhibition.weight)
            for i in range(self.dendrite_layers_n - 1, -1, -1):
                self.dendrite_layers[i].inputs_to_dendrite_branch.weight.data = torch.maximum(self.dendrite_layers[i].inputs_to_dendrite_branch.weight, 0 * self.dendrite_layers[i].inputs_to_dendrite_branch.weight)
                self.dendrite_layers[i].inhibition_to_dendrite_branch.weight.data = torch.maximum(self.dendrite_layers[i].inhibition_to_dendrite_branch.weight, 0 * self.dendrite_layers[i].inhibition_to_dendrite_branch.weight)

    def forward(self, input, labels):
        input = input.view(input.size(0), -1)  # Flatten the input tensor
        self.inhibition_act = self.inputs_to_inhibition(input)
        if self.shunting:
            self.inhibition_out = self.inhibition_act / (1 + self.inhibition_act)
        else:
            self.inhibition_out = self.sigmoid(self.inhibition_act)
        
        self.dendrite_branch_out = [None] * self.dendrite_layers_n

        for i in range(self.dendrite_layers_n - 1, -1, -1):
            is_outermost_layer = (i == (self.dendrite_layers_n - 1))
            this_dendrite_branch_out = None if is_outermost_layer else self.dendrite_branch_out[i + 1]
            self.dendrite_branch_out[i], self.inhibitory_inputs[i], self.excitatory_inputs[i] = self.dendrite_layers[i](input, self.inhibition_out, this_dendrite_branch_out)
            self.dendrite_activations[i] = self.dendrite_branch_out[i]
            
            # Apply attention mechanism
            self.dendrite_branch_out[i] = self.attention_layers[i](self.dendrite_branch_out[i])

        self.soma_act = self.dendrite_to_soma(self.dendrite_branch_out[0])
        return self.soma_act, labels

    def l1_loss(self):
        l1_loss = 0
        for attention_layer in self.attention_layers:
            l1_loss += torch.norm(attention_layer.weight, 1)
        return self.lambda_l1 * l1_loss
    

    
# Remove attention layers from the fully trained network with attention
class NetWithoutAttention(nn.Module):
    def __init__(self, net_with_attention):
        super(NetWithoutAttention, self).__init__()
        self.inputs_to_inhibition = net_with_attention.inputs_to_inhibition
        self.dendrite_layers = net_with_attention.dendrite_layers
        self.dendrite_to_soma = net_with_attention.dendrite_to_soma
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, input):
        self.inhibition_act = self.inputs_to_inhibition(input)
        if shunting:
            self.inhibition_out = self.inhibition_act / (1 + self.inhibition_act)
        else:
            self.inhibition_out = self.sigmoid(self.inhibition_act)

        self.dendrite_branch_out = [None] * dendrite_layers_n

        for i in range(dendrite_layers_n - 1, -1, -1):
            is_outermost_layer = (i == (dendrite_layers_n - 1))
            this_dendrite_branch_out = None if is_outermost_layer else self.dendrite_branch_out[i + 1]
            self.dendrite_branch_out[i] = self.dendrite_layers[i](input, self.inhibition_out, this_dendrite_branch_out)

        self.soma_act = self.dendrite_to_soma(self.dendrite_branch_out[0])
        return self.soma_act

Define Training and Evaluation Functions

In [6]:
def calculate_effective_coefficients(net):
    effective_coefficients = []

    # Initialize effective coefficients with the same values as the coefficients in the outermost layer
    for layer_idx in range(net.dendrite_layers_n):
        effective_coefficients.append(net.dendrite_layers[layer_idx].coefficients.clone())

    # Iterate from the outermost layer towards the soma (innermost layer)
    for layer_idx in range(net.dendrite_layers_n - 2, -1, -1):
        for branch_idx, coeff in enumerate(net.dendrite_layers[layer_idx].coefficients):
            # Multiply current branch's coefficient by the effective coefficients of its children branches
            effective_coefficients[layer_idx][branch_idx] *= effective_coefficients[layer_idx + 1][branch_idx]

    return effective_coefficients


def train_network(net, train_gen, test_gen, num_epochs, lr, scheduler_gamma, l1_lambda=0.0, fine_tune_l1=False, compute_mutual_info_last_epoch_only=True):
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=scheduler_gamma)
    lossi = []
    accuracies = np.zeros((3, num_epochs))
    mutual_infos_soma = None
    mutual_infos_dendrite = None
    mutual_infos_inhibitory = None
    mutual_infos_excitatory = None

    # Initialize mutual information containers only if needed
    #if not compute_mutual_info_last_epoch_only:
    mutual_infos_soma = np.zeros((3, num_epochs, 10))
    mutual_infos_dendrite = []
    mutual_infos_inhibitory = []
    mutual_infos_excitatory = []
    for layer_idx in range(net.dendrite_layers_n):
        mutual_infos_dendrite.append(np.zeros((3, num_epochs, net.dendrite_layers[layer_idx].inputs_to_dendrite_branch.out_features)))
        mutual_infos_inhibitory.append(np.zeros((3, num_epochs, net.dendrite_layers[layer_idx].inhibition_to_dendrite_branch.out_features)))
        mutual_infos_excitatory.append(np.zeros((3, num_epochs, net.dendrite_layers[layer_idx].inputs_to_dendrite_branch.out_features)))

    for epoch in range(num_epochs):
        running_loss = 0.0
        correct_d = 0
        correct_d1 = 0
        correct_d2 = 0
        total = 0
        all_outputs = []
        all_labels_d = []
        all_labels_d1 = []
        all_labels_d2 = []

        # Reinitialize activations and inputs for mutual information if needed
        dendrite_activations = None
        inhibitory_inputs = None
        excitatory_inputs = None
        if not compute_mutual_info_last_epoch_only or epoch==num_epochs-1:
            dendrite_activations = [[] for _ in range(net.dendrite_layers_n)]
            inhibitory_inputs = [[] for _ in range(net.dendrite_layers_n)]
            excitatory_inputs = [[] for _ in range(net.dendrite_layers_n)]

        for i, (images_raw, labels_raw) in enumerate(train_gen):
            images = transform_images(images_raw)
            images = images.flatten(start_dim=1).to(device)
            #print(images.shape)
            #images = images_raw.view(-1, 28 * 28 * 2).to(device)
            labels_d = sum_labels(labels_raw).to(device)
            labels_d1 = labels_raw[index_list1].to(device)
            labels_d2 = labels_raw[index_list2].to(device)

            optimizer.zero_grad()
            outputs, _ = net(images, labels_d)
            loss = loss_function(outputs, labels_d)
            loss0 = loss.clone()

            # Apply L1 regularization based on effective coefficients
            if fine_tune_l1 and l1_lambda > 0:
                effective_coefficients = calculate_effective_coefficients(net)
                l1_reg = torch.tensor(0., requires_grad=True).to(device)
                for layer_idx in range(net.dendrite_layers_n):
                    l1_reg += torch.norm(effective_coefficients[layer_idx], 1)
                loss += l1_lambda * l1_reg

            loss.backward()
            optimizer.step()
            net.trimWeights()
            running_loss += loss0.item()

            # Compute accuracy
            _, predicted = torch.max(outputs, 1)
            correct_d += (predicted == labels_d).sum().item()
            correct_d1 += (predicted == labels_d1).sum().item()
            correct_d2 += (predicted == labels_d2).sum().item()
            total += labels_d.size(0)

            # Collect outputs and labels for mutual information calculation
            if not compute_mutual_info_last_epoch_only or epoch==num_epochs-1:
                all_outputs.append(outputs.cpu().detach().numpy())
                all_labels_d.extend(labels_d.cpu().numpy())
                all_labels_d1.extend(labels_d1.cpu().numpy())
                all_labels_d2.extend(labels_d2.cpu().numpy())

                # Collect dendritic activations and inputs
                for layer_idx in range(net.dendrite_layers_n):
                    dendrite_activations[layer_idx].append(net.dendrite_activations[layer_idx].cpu().detach().numpy())
                    inhibitory_inputs[layer_idx].append(net.inhibitory_inputs[layer_idx].cpu().detach().numpy())
                    excitatory_inputs[layer_idx].append(net.excitatory_inputs[layer_idx].cpu().detach().numpy())

        scheduler.step()
        avg_loss = running_loss / len(train_gen)
        accuracy_d = 100 * correct_d / total
        accuracy_d1 = 100 * correct_d1 / total
        accuracy_d2 = 100 * correct_d2 / total
        lossi.append(avg_loss)
        accuracies[0, epoch] = accuracy_d
        accuracies[1, epoch] = accuracy_d1
        accuracies[2, epoch] = accuracy_d2
        
        # Mutual information calculation in the last epoch or during training
        if (epoch == num_epochs - 1) or not compute_mutual_info_last_epoch_only:
            if len(all_outputs) > 0:  # Check if we have collected any outputs
                all_outputs = np.concatenate(all_outputs)
                all_labels_d = np.array(all_labels_d)
                all_labels_d1 = np.array(all_labels_d1)
                all_labels_d2 = np.array(all_labels_d2)

                # Compute mutual information for the current epoch
                mi_d = compute_mutual_information(all_outputs, all_labels_d)
                mi_d1 = compute_mutual_information(all_outputs, all_labels_d1)
                mi_d2 = compute_mutual_information(all_outputs, all_labels_d2)
                if mutual_infos_soma is not None:
                    mutual_infos_soma[0, epoch, :] = mi_d
                    mutual_infos_soma[1, epoch, :] = mi_d1
                    mutual_infos_soma[2, epoch, :] = mi_d2

                for layer_idx in range(net.dendrite_layers_n):
                    dend_activations = np.concatenate(dendrite_activations[layer_idx])
                    inh_inputs = np.concatenate(inhibitory_inputs[layer_idx])
                    exc_inputs = np.concatenate(excitatory_inputs[layer_idx])

                    mi_dend = compute_mutual_information(dend_activations, all_labels_d)
                    mi_inhibitory = compute_mutual_information(inh_inputs, all_labels_d)
                    mi_excitatory = compute_mutual_information(exc_inputs, all_labels_d)

                    if mutual_infos_dendrite is not None:
                        mutual_infos_dendrite[layer_idx][0, epoch, :] = mi_dend
                        mutual_infos_inhibitory[layer_idx][0, epoch, :] = mi_inhibitory
                        mutual_infos_excitatory[layer_idx][0, epoch, :] = mi_excitatory

                        mi_dend = compute_mutual_information(dend_activations, all_labels_d1)
                        mi_inhibitory = compute_mutual_information(inh_inputs, all_labels_d1)
                        mi_excitatory = compute_mutual_information(exc_inputs, all_labels_d1)

                        mutual_infos_dendrite[layer_idx][1, epoch, :] = mi_dend
                        mutual_infos_inhibitory[layer_idx][1, epoch, :] = mi_inhibitory
                        mutual_infos_excitatory[layer_idx][1, epoch, :] = mi_excitatory

                        mi_dend = compute_mutual_information(dend_activations, all_labels_d2)
                        mi_inhibitory = compute_mutual_information(inh_inputs, all_labels_d2)
                        mi_excitatory = compute_mutual_information(exc_inputs, all_labels_d2)

                        mutual_infos_dendrite[layer_idx][2, epoch, :] = mi_dend
                        mutual_infos_inhibitory[layer_idx][2, epoch, :] = mi_inhibitory
                        mutual_infos_excitatory[layer_idx][2, epoch, :] = mi_excitatory
            
                print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy d: {accuracy_d:.2f}%, d1: {accuracy_d1:.2f}%, d2: {accuracy_d2:.2f}%, Mutual Information d: {np.mean(mi_d):.4f}, d1: {np.mean(mi_d1):.4f}, d2: {np.mean(mi_d2):.4f}')
                
                print(all_outputs)
                    
                
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}')

    return net, lossi, accuracies, mutual_infos_soma, mutual_infos_dendrite, mutual_infos_inhibitory, mutual_infos_excitatory

def evaluate_network(net, test_gen):
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    for images_raw, labels_raw in test_gen:
        images = transform_images(images_raw.view(-1, 28*28)).to(device)
        labels = sum_labels(labels_raw).to(device)

        with torch.no_grad():
            output = net(images)
            _, predicted = torch.max(output, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = (100 * correct) / total
    print(f'Accuracy of the model: {accuracy:.3f} %')

    return accuracy, all_preds, all_labels

def plot_confusion_matrix(all_labels, all_preds, accuracy):
    from sklearn.metrics import confusion_matrix
    import seaborn as sns

    conf_matrix = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=np.arange(10), yticklabels=np.arange(10))
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix\nAccuracy of the model: {accuracy:.3f} %')
    plt.show()

In [7]:
# Build the models
net_without_attention = Net(input_size, inhibitory_units_n, synapses_per_inhibitory_unit, dendrite_layers_n, dendrite_branch_n, 
                            inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, shunting, soma_n).to(device)
net_without_attention.initWeights()

net_without_attention, lossi, accuracies, mutual_infos_soma, mutual_infos_dendrite, mutual_infos_inhibitory, mutual_infos_excitatory = train_network(
    net_without_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma, l1_lambda=0.0, fine_tune_l1=False, compute_mutual_info_last_epoch_only=True)

torch.save(net_without_attention.state_dict(), 'model_without_l1.pth')

net_without_attention.load_state_dict(torch.load('model_without_l1.pth', weights_only=True))
net_without_attention, lossi, accuracies, mutual_infos_soma, mutual_infos_dendrite, mutual_infos_inhibitory, mutual_infos_excitatory = train_network(
    net_without_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma, l1_lambda=1, fine_tune_l1=True, compute_mutual_info_last_epoch_only=True)

Epoch [1/200], Loss: 2.3006
Epoch [2/200], Loss: 2.2887
Epoch [3/200], Loss: 2.2805
Epoch [4/200], Loss: 2.2759
Epoch [5/200], Loss: 2.2736
Epoch [6/200], Loss: 2.2719
Epoch [7/200], Loss: 2.2706
Epoch [8/200], Loss: 2.2698
Epoch [9/200], Loss: 2.2685
Epoch [10/200], Loss: 2.2687
Epoch [11/200], Loss: 2.2679
Epoch [12/200], Loss: 2.2669
Epoch [13/200], Loss: 2.2653
Epoch [14/200], Loss: 2.2643
Epoch [15/200], Loss: 2.2645
Epoch [16/200], Loss: 2.2641
Epoch [17/200], Loss: 2.2640
Epoch [18/200], Loss: 2.2638
Epoch [19/200], Loss: 2.2635
Epoch [20/200], Loss: 2.2631
Epoch [21/200], Loss: 2.2635
Epoch [22/200], Loss: 2.2624
Epoch [23/200], Loss: 2.2638
Epoch [24/200], Loss: 2.2621
Epoch [25/200], Loss: 2.2617
Epoch [26/200], Loss: 2.2609
Epoch [27/200], Loss: 2.2613
Epoch [28/200], Loss: 2.2606
Epoch [29/200], Loss: 2.2601
Epoch [30/200], Loss: 2.2603
Epoch [31/200], Loss: 2.2601
Epoch [32/200], Loss: 2.2599
Epoch [33/200], Loss: 2.2593
Epoch [34/200], Loss: 2.2591
Epoch [35/200], Loss: 2

KeyboardInterrupt: 

In [ ]:
# Build the models
#net_without_attention = Net(input_size, inhibitory_units_n, synapses_per_inhibitory_unit, dendrite_layers_n, dendrite_branch_n, 
#                            inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, shunting, soma_n).to(device)
#net_without_attention.initWeights()

# Update the main code to call the updated train_network function
#net_without_attention, lossi_without_attention, accuracies_without_attention, mutual_infos_soma, mutual_infos_dendrite, mutual_infos_inhibitory, mutual_infos_excitatory = train_network(net_without_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma, use_l1_regularization=False)

# Plotting mutual information for soma
mean_mutual_infos_soma = np.mean(mutual_infos_soma, axis=2)
plt.figure()
plt.plot(mean_mutual_infos_soma[0, :], label='MI d')
plt.plot(mean_mutual_infos_soma[1, :], label='MI d1')
plt.plot(mean_mutual_infos_soma[2, :], label='MI d2')
plt.xlabel('Epoch')
plt.ylabel('Mean Mutual Information')
plt.title('Mean Mutual Information for Soma Outputs')
plt.legend()
plt.show()

# Plotting mutual information for dendritic activations
for layer_idx in range(net_without_attention.dendrite_layers_n):
    mean_mutual_infos_dendrite = np.mean(mutual_infos_dendrite[layer_idx], axis=2)
    plt.figure()
    plt.plot(mean_mutual_infos_dendrite[0, :], label=f'MI d - Layer {layer_idx + 1}')
    plt.plot(mean_mutual_infos_dendrite[1, :], label=f'MI d1 - Layer {layer_idx + 1}')
    plt.plot(mean_mutual_infos_dendrite[2, :], label=f'MI d2 - Layer {layer_idx + 1}')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Mutual Information')
    plt.title(f'Mean Mutual Information for Dendritic Activations - Layer {layer_idx + 1}')
    plt.legend()
    plt.show()

# Plotting mutual information for inhibitory inputs
for layer_idx in range(net_without_attention.dendrite_layers_n):
    mean_mutual_infos_inhibitory = np.mean(mutual_infos_inhibitory[layer_idx], axis=2)
    plt.figure()
    plt.plot(mean_mutual_infos_inhibitory[0, :], label=f'MI d - Layer {layer_idx + 1}')
    plt.plot(mean_mutual_infos_inhibitory[1, :], label=f'MI d1 - Layer {layer_idx + 1}')
    plt.plot(mean_mutual_infos_inhibitory[2, :], label=f'MI d2 - Layer {layer_idx + 1}')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Mutual Information')
    plt.title(f'Mean Mutual Information for Inhibitory Inputs - Layer {layer_idx + 1}')
    plt.legend()
    plt.show()

# Plotting mutual information for excitatory inputs
for layer_idx in range(net_without_attention.dendrite_layers_n):
    mean_mutual_infos_excitatory = np.mean(mutual_infos_excitatory[layer_idx], axis=2)
    plt.figure()
    plt.plot(mean_mutual_infos_excitatory[0, :], label=f'MI d - Layer {layer_idx + 1}')
    plt.plot(mean_mutual_infos_excitatory[1, :], label=f'MI d1 - Layer {layer_idx + 1}')
    plt.plot(mean_mutual_infos_excitatory[2, :], label=f'MI d2 - Layer {layer_idx + 1}')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Mutual Information')
    plt.title(f'Mean Mutual Information for Excitatory Inputs - Layer {layer_idx + 1}')
    plt.legend()
    plt.show()




In [ ]:


def extract_pruned_tree(net):
    pruned_tree = {}
    
    for layer_idx in range(net.dendrite_layers_n):
        layer = net.dendrite_layers[layer_idx]
        coefficients = layer.coefficients.detach().cpu().numpy()
        non_zero_indices = np.where(coefficients != 0)[0]
        
        pruned_tree[layer_idx] = non_zero_indices.tolist()
        
        print(f"Layer {layer_idx + 1}: {len(non_zero_indices)} non-zero dendrites out of {len(coefficients)}")
        
    return pruned_tree

# Call this function after training is completed
pruned_tree = extract_pruned_tree(net_without_attention)

# Optionally, you can visualize the tree using your existing tree visualization code or simply print it out
print("Pruned Dendritic Tree Structure:", pruned_tree)


# Plot the pruned tree
#plot_pruned_dendritic_tree(pruned_tree)

import matplotlib.pyplot as plt
import networkx as nx

def plot_pruned_dendritic_tree(pruned_tree):
    G = nx.DiGraph()

    # Add nodes and edges based on the pruned tree
    previous_layer_nodes = ['Input']
    G.add_node('Input', layer=0)  # Adding the input node with layer 0

    for layer_idx, active_dendrites in pruned_tree.items():
        current_layer_nodes = []
        for dendrite_idx in active_dendrites:
            node_name = ''#f'Layer{layer_idx}_Branch{dendrite_idx+1}'
            G.add_node(node_name, layer=layer_idx + 1)  # Adding layer information
            current_layer_nodes.append(node_name)

            # Connect nodes from the previous layer to the current node
            for prev_node in previous_layer_nodes:
                G.add_edge(prev_node, node_name)

        # Update the previous layer nodes to the current layer
        previous_layer_nodes = current_layer_nodes

    # Connect the last layer to the Soma
    G.add_node('Soma', layer=len(pruned_tree) + 1)  # Adding the Soma node with the last layer index
    for last_node in previous_layer_nodes:
        G.add_edge(last_node, 'Soma')

    # Draw the graph
    pos = nx.multipartite_layout(G, subset_key="layer")
    plt.figure(figsize=(10, 8))
    nx.draw(G, pos, with_labels=True, node_size=2000, node_color="lightblue", font_size=10, font_weight="bold", edge_color="gray")
    plt.title("Pruned Dendritic Tree Visualization")
    plt.show()

# Example: Assuming pruned_tree is a dictionary with pruned branches
pruned_tree_example = {
    1: [0, 1],  # Both branches in Layer 1 are active
    2: [0]      # Only the first branch in Layer 2 is active
}

# Plot the pruned tree
plot_pruned_dendritic_tree(pruned_tree)



In [ ]:
layer = net_without_attention.dendrite_layers[1]
coefficients = layer.coefficients.detach().cpu().numpy()
print(layer)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from networkx.drawing.nx_agraph import graphviz_layout

def plot_dendritic_tree(net, threshold=0.1):
    G = nx.DiGraph()

    # Traverse the network and add nodes and edges
    for layer_idx in range(net.dendrite_layers_n):
        num_branches = net.dendrite_layers[layer_idx].inputs_to_dendrite_branch.out_features  # Assuming this is the correct attribute
        for branch_idx in range(num_branches):
            node_name = ''#f"L{layer_idx+1}_B{branch_idx+1}"
            G.add_node(node_name, layer=layer_idx)
            
            if layer_idx > 0:
                prev_num_branches = net.dendrite_layers[layer_idx - 1].inputs_to_dendrite_branch.out_features
                for prev_branch_idx in range(prev_num_branches):
                    prev_node_name = ''#f"L{layer_idx}_B{prev_branch_idx+1}"
                    weight = net.dendrite_layers[layer_idx].inputs_to_dendrite_branch.weight[branch_idx, prev_branch_idx].item()
                    if abs(weight) > threshold:  # Prune low-weight connections
                        G.add_edge(prev_node_name, node_name, weight=weight)

    # Add the soma as the final node
    G.add_node("Soma", layer=net.dendrite_layers_n)
    num_branches_last_layer = net.dendrite_layers[-1].inputs_to_dendrite_branch.out_features
    for branch_idx in range(num_branches_last_layer):
        node_name = ''#f"L{net.dendrite_layers_n}_B{branch_idx+1}"
        G.add_edge(node_name, "Soma", weight=1.0)  # Assume strong connection to Soma

    # Layout using Graphviz's 'dot' for tree structure
    pos = graphviz_layout(G, prog='dot')

    # Draw the graph
    plt.figure(figsize=(12, 10))
    nx.draw(G, pos, with_labels=True, node_size=800, node_color='skyblue', font_size=10, font_weight='bold', edge_color='gray')
    nx.draw_networkx_edge_labels(G, pos, edge_labels={(u, v): f'{d["weight"]:.2f}' for u, v, d in G.edges(data=True)})
    plt.title("Dendritic Tree Visualization")
    plt.show()

# Call this function to visualize the network
plot_dendritic_tree(net_without_attention, threshold=0.1)

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt

def plot_attention_weights(net):
    for layer_idx, attention_layer in enumerate(net.attention_layers):
        weights = attention_layer.weight.detach().cpu().numpy()
        plt.figure(figsize=(10, 8))
        sns.heatmap(weights, cmap='viridis')
        plt.title(f'Attention Weights - Layer {layer_idx + 1}')
        plt.xlabel('Dendritic Branches')
        plt.ylabel('Dendritic Branches')
        plt.show()



# Training call
net_with_attention = NetWithAttention(input_size, inhibitory_units_n, synapses_per_inhibitory_unit, dendrite_layers_n, dendrite_branch_n, 
                                      inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, shunting, soma_n).to(device)
net_with_attention.initWeights()
net_with_attention, lossi_with_attention, accuracies_with_attention, mutual_infos_soma_with_attention, mutual_infos_dendrite_with_attention, mutual_infos_inhibitory_with_attention, mutual_infos_excitatory_with_attention = train_network(net_with_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma)

# Visualize attention weights
plot_attention_weights(net_with_attention)

In [ ]:
# Plotting mutual information for soma
mean_mutual_infos_soma = np.mean(mutual_infos_soma, axis=2)
plt.figure()
plt.plot(mean_mutual_infos_soma[0, :], label='MI d')
plt.plot(np.mean(mean_mutual_infos_soma[1:2, :],0), label='MI di')
#plt.plot(mean_mutual_infos_soma[2, :], label='MI d2')
plt.xlabel('Epoch')
plt.ylabel('Mean Mutual Information')
plt.title('Mean Mutual Information for Soma Outputs')
plt.legend()
plt.show()
info_lay_d = np.zeros((net_without_attention.dendrite_layers_n, ), dtype=float)
info_lay_di = np.zeros((net_without_attention.dendrite_layers_n, ), dtype=float)
# Plotting mutual information for dendritic activations
for layer_idx in range(net_without_attention.dendrite_layers_n):
    mean_mutual_infos_dendrite = np.mean(mutual_infos_dendrite[layer_idx], axis=2)
    info_lay_d[layer_idx] = mean_mutual_infos_dendrite[0,-1]
    info_lay_di[layer_idx] = np.mean(mean_mutual_infos_dendrite[1:2,-1],0)
    
    plt.figure()
    plt.plot(mean_mutual_infos_dendrite[0, :], label=f'MI d - Layer {layer_idx + 1}')
    plt.plot(np.mean(mean_mutual_infos_dendrite[1:2, :],0), label=f'MI d1 - Layer {layer_idx + 1}')
    #plt.plot(mean_mutual_infos_dendrite[2, :], label=f'MI d2 - Layer {layer_idx + 1}')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Mutual Information')
    plt.title(f'Mean Mutual Information for Dendritic Activations - Layer {layer_idx + 1}')
    plt.legend()
    plt.show()
    
    
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot information about d
axs[0].bar(range(num_layers), info_lay_d, color='b')
axs[0].set_title('Information about d')
axs[0].set_xlabel('Layer Depth-dendrites')
axs[0].set_ylabel('Mutual Information')
axs[0].set_xticks(range(num_layers))
axs[0].set_xticklabels([f'Layer {i+1}' for i in range(num_layers)])
axs[0].set_ylim(0, 0.0014)

# Plot information about di
axs[1].bar(range(num_layers), info_lay_di, color='r')
axs[1].set_title('Information about di')
axs[1].set_xlabel('Layer Depthdendrites')
axs[1].set_ylabel('Mutual Information')
axs[1].set_xticks(range(num_layers))
axs[1].set_xticklabels([f'Layer {i+1}' for i in range(num_layers)])
axs[1].set_ylim(0, 0.0014)

# Adjust layout
plt.tight_layout()

# Show plot
plt.show()


# Plotting mutual information for inhibitory inputs
for layer_idx in range(net_without_attention.dendrite_layers_n):
    mean_mutual_infos_inhibitory = np.mean(mutual_infos_inhibitory[layer_idx], axis=2)
    info_lay_d[layer_idx] = mean_mutual_infos_inhibitory[0,-1]
    info_lay_di[layer_idx] = np.mean(mean_mutual_infos_inhibitory[1:2,-1],0)
    plt.figure()
    plt.plot(mean_mutual_infos_inhibitory[0, :], label=f'MI d - Layer {layer_idx + 1}')
    plt.plot(np.mean(mean_mutual_infos_inhibitory[1:2, :],0), label=f'MI di - Layer {layer_idx + 1}')
    #plt.plot(mean_mutual_infos_inhibitory[2, :], label=f'MI d2 - Layer {layer_idx + 1}')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Mutual Information')
    plt.title(f'Mean Mutual Information for Inhibitory Inputs - Layer {layer_idx + 1}')
    plt.legend()
    plt.show()

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot information about d
axs[0].bar(range(num_layers), info_lay_d, color='b')
axs[0].set_title('Information about d')
axs[0].set_xlabel('Layer Depth-inhibitory inputs')
axs[0].set_ylabel('Mutual Information')
axs[0].set_xticks(range(num_layers))
axs[0].set_xticklabels([f'Layer {i+1}' for i in range(num_layers)])
axs[0].set_ylim(0, 0.0014)

# Plot information about di
axs[1].bar(range(num_layers), info_lay_di, color='r')
axs[1].set_title('Information about di')
axs[1].set_xlabel('Layer Depth-inhibitory inputs')
axs[1].set_ylabel('Mutual Information')
axs[1].set_xticks(range(num_layers))
axs[1].set_xticklabels([f'Layer {i+1}' for i in range(num_layers)])
axs[1].set_ylim(0, 0.0014)

# Adjust layout
plt.tight_layout()

# Show plot
plt.show()

# Plotting mutual information for excitatory inputs
for layer_idx in range(net_without_attention.dendrite_layers_n):
    mean_mutual_infos_excitatory = np.mean(mutual_infos_excitatory[layer_idx], axis=2)
    info_lay_d[layer_idx] = mean_mutual_infos_excitatory[0,-1]
    info_lay_di[layer_idx] = np.mean(mean_mutual_infos_excitatory[1:2,-1],0)

    plt.figure()
    plt.plot(mean_mutual_infos_excitatory[0, :], label=f'MI d - Layer {layer_idx + 1}')
    plt.plot(np.mean(mean_mutual_infos_excitatory[1:2, :],0), label=f'MI d1 - Layer {layer_idx + 1}')
    #plt.plot(mean_mutual_infos_excitatory[2, :], label=f'MI d2 - Layer {layer_idx + 1}')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Mutual Information')
    plt.title(f'Mean Mutual Information for Excitatory Inputs - Layer {layer_idx + 1}')
    plt.legend()
    plt.show()
    
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot information about d
axs[0].bar(range(num_layers), info_lay_d, color='b')
axs[0].set_title('Information about d')
axs[0].set_xlabel('Layer Depth-excitatory inputs')
axs[0].set_ylabel('Mutual Information')
axs[0].set_xticks(range(num_layers))
axs[0].set_xticklabels([f'Layer {i+1}' for i in range(num_layers)])
axs[0].set_ylim(0, 0.0014)

# Plot information about di
axs[1].bar(range(num_layers), info_lay_di, color='r')
axs[1].set_title('Information about di')
axs[1].set_xlabel('Layer Depth-excitatory inputs')
axs[1].set_ylabel('Mutual Information')
axs[1].set_xticks(range(num_layers))
axs[1].set_xticklabels([f'Layer {i+1}' for i in range(num_layers)])
axs[1].set_ylim(0, 0.0014)

# Adjust layout
plt.tight_layout()

# Show plot
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Helper function to ensure consistent array shapes
def pad_and_stack(arrays):
    max_length = max(len(arr) for arr in arrays)
    padded_arrays = [np.pad(arr, ((0, 0), (0, max_length - len(arr))), mode='constant', constant_values=0) for arr in arrays]
    return np.stack(padded_arrays, axis=0)
# Ensure consistent shapes for mutual information arrays
mutual_infos_dendrite_padded = [pad_and_stack(layer) for layer in mutual_infos_dendrite]
mutual_infos_inhibitory_padded = [pad_and_stack(layer) for layer in mutual_infos_inhibitory]
mutual_infos_excitatory_padded = [pad_and_stack(layer) for layer in mutual_infos_excitatory]

# Helper function to extract the mean values for the last epoch
def get_mean_last_epoch(data, layer_count):
    last_epoch_data = []
    for layer_idx in range(layer_count):
        mean_data = np.mean(data[layer_idx], axis=1)
        last_epoch_data.append(mean_data[:, -1])  # Get the last epoch
    return np.array(last_epoch_data).T  # Transpose to match the required format

# Extract mean mutual information for the last epoch
mean_mutual_infos_soma = np.mean(mutual_infos_soma, axis=2)[:, -1]
mean_mutual_infos_dendrite_last = get_mean_last_epoch(mutual_infos_dendrite_padded, net_without_attention.dendrite_layers_n)
mean_mutual_infos_inhibitory_last = get_mean_last_epoch(mutual_infos_inhibitory_padded, net_without_attention.dendrite_layers_n)
mean_mutual_infos_excitatory_last = get_mean_last_epoch(mutual_infos_excitatory_padded, net_without_attention.dendrite_layers_n)

# Plotting mutual information for soma
plt.figure()
bar_width = 0.35
index = np.arange(1)
plt.bar(index, mean_mutual_infos_soma[0], bar_width, label='MI d')
plt.bar(index + bar_width, mean_mutual_infos_soma[1], bar_width, label='MI di')
plt.xlabel('Soma Outputs')
plt.ylabel('Mean Mutual Information')
plt.title('Mean Mutual Information for Soma Outputs (Last Epoch)')
plt.legend()
plt.savefig('mean_mutual_information_soma_bar.png')
plt.show()

# Plotting mutual information for dendritic activations
plt.figure()
bar_width = 0.35
index = np.arange(net_without_attention.dendrite_layers_n)
plt.bar(index, mean_mutual_infos_dendrite_last[0, :], bar_width, label='MI d')
plt.bar(index + bar_width, mean_mutual_infos_dendrite_last[1, :], bar_width, label='MI di')
plt.xlabel('Layer')
plt.ylabel('Mean Mutual Information')
plt.title('Mean Mutual Information for Dendritic Activations (Last Epoch)')
plt.legend()
plt.savefig('mean_mutual_information_dendritic_activations_bar.png')
plt.show()

# Plotting mutual information for inhibitory and excitatory inputs
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
bar_width = 0.35
index = np.arange(net_without_attention.dendrite_layers_n)

axs[0].bar(index, mean_mutual_infos_inhibitory_last[0, :], bar_width, label='MI d')
axs[0].bar(index + bar_width, mean_mutual_infos_inhibitory_last[1, :], bar_width, label='MI di')
axs[0].set_xlabel('Layer')
axs[0].set_ylabel('Mean Mutual Information')
axs[0].set_title('Mean Mutual Information for Inhibitory Inputs (Last Epoch)')
axs[0].legend()

axs[1].bar(index, mean_mutual_infos_excitatory_last[0, :], bar_width, label='MI d')
axs[1].bar(index + bar_width, mean_mutual_infos_excitatory_last[1, :], bar_width, label='MI di')
axs[1].set_xlabel('Layer')
axs[1].set_ylabel('Mean Mutual Information')
axs[1].set_title('Mean Mutual Information for Excitatory Inputs (Last Epoch)')
axs[1].legend()

plt.savefig('mean_mutual_information_inhibitory_excitatory_inputs_bar.png')
plt.show()

In [ ]:
print(mean_mutual_infos_inhibitory_last.shape)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plotting mutual information for soma
mean_mutual_infos_soma = np.mean(mutual_infos_soma, axis=2)
plt.figure()
plt.plot(mean_mutual_infos_soma[0, :], label='MI d')
plt.plot(np.mean(mean_mutual_infos_soma[1:2, :], 0), label='MI di')
plt.xlabel('Epoch')
plt.ylabel('Mean Mutual Information')
plt.title('Mean Mutual Information for Soma Outputs')
plt.legend()
plt.savefig('mean_mutual_information_soma.png')
plt.show()

# Plotting mutual information for dendritic activations
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

for layer_idx in range(net_without_attention.dendrite_layers_n):
    mean_mutual_infos_dendrite = np.mean(mutual_infos_dendrite[layer_idx], axis=2)
    axs[0].plot(mean_mutual_infos_dendrite[0, :], label=f'MI d - Layer {layer_idx + 1}')
    axs[1].plot(np.mean(mean_mutual_infos_dendrite[1:2, :], 0), label=f'MI di - Layer {layer_idx + 1}')

for ax in axs:
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Mean Mutual Information')
    ax.legend()

axs[0].set_title('Mean Mutual Information for Dendritic Activations (MI d)')
axs[1].set_title('Mean Mutual Information for Dendritic Activations (MI di)')
plt.savefig('mean_mutual_information_dendritic_activations.png')
plt.show()

# Plotting mutual information for inhibitory inputs
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

for layer_idx in range(net_without_attention.dendrite_layers_n):
    mean_mutual_infos_inhibitory = np.mean(mutual_infos_inhibitory[layer_idx], axis=2)
    axs[0].plot(mean_mutual_infos_inhibitory[0, :], label=f'MI d - Layer {layer_idx + 1}')
    axs[1].plot(np.mean(mean_mutual_infos_inhibitory[1:2, :], 0), label=f'MI di - Layer {layer_idx + 1}')

for ax in axs:
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Mean Mutual Information')
    ax.legend()

axs[0].set_title('Mean Mutual Information for Inhibitory Inputs (MI d)')
axs[1].set_title('Mean Mutual Information for Inhibitory Inputs (MI di)')
plt.savefig('mean_mutual_information_inhibitory_inputs.png')
plt.show()

# Plotting mutual information for excitatory inputs
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

for layer_idx in range(net_without_attention.dendrite_layers_n):
    mean_mutual_infos_excitatory = np.mean(mutual_infos_excitatory[layer_idx], axis=2)
    axs[0].plot(mean_mutual_infos_excitatory[0, :], label=f'MI d - Layer {layer_idx + 1}')
    axs[1].plot(np.mean(mean_mutual_infos_excitatory[1:2, :], 0), label=f'MI di - Layer {layer_idx + 1}')

for ax in axs:
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Mean Mutual Information')
    ax.legend()

axs[0].set_title('Mean Mutual Information for Excitatory Inputs (MI d)')
axs[1].set_title('Mean Mutual Information for Excitatory Inputs (MI di)')
plt.savefig('mean_mutual_information_excitatory_inputs.png')
plt.show()

In [ ]:
 
# Function to plot mutual information for each type
def plot_mutual_information_per_layer(mutual_infos, title, ylabel):
    fig, axes = plt.subplots(1, net_without_attention.dendrite_layers_n, figsize=(20, 5), sharey=True)
    for layer_idx in range(net_without_attention.dendrite_layers_n):
        for epoch in [0, num_epochs//2, num_epochs-1]:  # First, middle, and last epoch
            axes[layer_idx].plot(mutual_infos[layer_idx][0, epoch, :], label=f'MI d - Epoch {epoch + 1}')
            axes[layer_idx].plot(mutual_infos[layer_idx][1, epoch, :], label=f'MI d1 - Epoch {epoch + 1}')
            axes[layer_idx].plot(mutual_infos[layer_idx][2, epoch, :], label=f'MI d2 - Epoch {epoch + 1}')
        axes[layer_idx].set_title(f'Layer {layer_idx + 1}')
        axes[layer_idx].set_xlabel('Neuron Index')
    axes[0].set_ylabel(ylabel)
    fig.suptitle(title)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper right')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

# Plotting mutual information for soma
mean_mutual_infos_soma = np.mean(mutual_infos_soma, axis=2)
plt.figure()
plt.plot(mean_mutual_infos_soma[0, :], label='MI d - Soma')
plt.plot(mean_mutual_infos_soma[1, :], label='MI d1 - Soma')
plt.plot(mean_mutual_infos_soma[2, :], label='MI d2 - Soma')
plt.xlabel('Epoch')
plt.ylabel('Mean Mutual Information')
plt.title('Mean Mutual Information for Soma')
plt.legend()
plt.show()

# Plotting mutual information for dendritic activations
plot_mutual_information_per_layer(mutual_infos_dendrite, 'Mean Mutual Information for Dendritic Activations', 'Mean Mutual Information')

# Plotting mutual information for inhibitory inputs
plot_mutual_information_per_layer(mutual_infos_inhibitory, 'Mean Mutual Information for Inhibitory Inputs', 'Mean Mutual Information')

# Plotting mutual information for excitatory inputs
plot_mutual_information_per_layer(mutual_infos_excitatory, 'Mean Mutual Information for Excitatory Inputs', 'Mean Mutual Information')

TEST MI

In [ ]:

# Build the models
net_without_attention = Net(input_size, inhibitory_units_n, synapses_per_inhibitory_unit, dendrite_layers_n, dendrite_branch_n, 
                            inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, shunting, soma_n).to(device)
net_without_attention.initWeights()

# Train and evaluate the network without attention
net_without_attention, lossi_without_attention, accuracies_without_attention, information_without_attention = train_network(net_without_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma)

# Plotting mutual information
mutual_infos = np.array(information_without_attention)
accuraicies = np.array(accuracies_without_attention)

n = 3
fig, ax = plt.subplots(1, 2, figsize=(15, 5))  # 1 row, 2 columns

# First subplot
ax[0].plot(mutual_infos[0, :, n], label='Mutual Information d')
ax[0].plot(mutual_infos[1, :, n], label='Mutual Information d1')
ax[0].plot(mutual_infos[2, :, n], label='Mutual Information d2')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Mutual Information')
ax[0].legend()
ax[0].set_title('Mutual Information over Epochs')

# Second subplot
ax[1].plot(accuraicies[0, :], label='Accuracy d')
ax[1].plot(accuraicies[1, :], label='Accuracy d1')
ax[1].plot(accuraicies[2, :], label='Accuracy d2')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Accuracy')
ax[1].legend()
ax[1].set_title('Accuracy over Epochs')

plt.tight_layout()  # Adjust subplots to fit into the figure area.
plt.show()

In [ ]:
n =0
fig, ax = plt.subplots(1, 2, figsize=(15, 5))  # 1 row, 2 columns
accuraicies = np.array(accuracies_without_attention)
mutual_infos = np.array(information_without_attention)

# First subplot
ax[0].plot(mutual_infos[0, :, 1:],color='b')
ax[0].plot(mutual_infos[1, :, 1:],color='r')
ax[0].plot(mutual_infos[2, :, 1:],color='m')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Mutual Information')
ax[0].plot(mutual_infos[0, :, 0], color='b', label='Mutual Information d')
ax[0].plot(mutual_infos[1, :, 0], color='r', label='Mutual Information d1')
ax[0].plot(mutual_infos[2, :, 0], color='m', label='Mutual Information d2')
ax[0].legend()
ax[0].set_title('Mutual Information over Epochs')

# Second subplot
ax[1].plot(accuraicies[0, :, 1:],color='b')
ax[1].plot(accuraicies[1, :, 1:],color='r')
ax[1].plot(accuraicies[2, :, 1:],color='m')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Accuracy')
#ax[1].legend()
ax[1].plot(accuraicies[0, :, 0], color='b',label='Accuracy d')
ax[1].plot(accuraicies[1, :, 0], color='r',label='Accuracy d1')
ax[1].plot(accuraicies[2, :, 0], color='m',label='Accuracy d2')
ax[1].legend()
ax[1].set_title('Accuracy over Epochs')

plt.tight_layout()  # Adjust subplots to fit into the figure area.
plt.show()


Train and Evaluate Network without Attention

In [ ]:
break

# Build the models
net_without_attention = Net(input_size, inhibitory_units_n, synapses_per_inhibitory_unit, dendrite_layers_n, dendrite_branch_n, 
                            inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, shunting, soma_n).to(device)
net_without_attention.initWeights()

net_with_attention = NetWithAttention(input_size, inhibitory_units_n, synapses_per_inhibitory_unit, dendrite_layers_n, dendrite_branch_n, 
                                      inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, shunting, soma_n).to(device)
net_with_attention.initWeights()

# Train and evaluate the network without attention
net_without_attention, lossi_without_attention, accuracies_without_attention, activations_by_epoch_without_attention = train_network(
    net_without_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma, capture_activations=True)
accuracy_without_attention, all_preds_without_attention, all_labels_without_attention = evaluate_network(net_without_attention, test_gen)
plot_confusion_matrix(all_labels_without_attention, all_preds_without_attention, accuracy_without_attention)

Train and Evaluate Attention Layers with Pre-trained Network

In [ ]:
# Load pre-trained weights into the network with attention
net_with_attention.load_state_dict(net_without_attention.state_dict(), strict=False)

# Freeze all parameters except for attention layers
for param in net_with_attention.parameters():
    param.requires_grad = False

for layer in net_with_attention.attention_layers:
    for param in layer.parameters():
        param.requires_grad = True

# Train and evaluate the attention layers
net_with_attention, lossi_attention_only, accuracies_attention_only, activations_by_epoch_attention_only = train_network(
    net_with_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma, capture_activations=True)

accuracy_attention_only, all_preds_attention_only, all_labels_attention_only = evaluate_network(net_with_attention, test_gen)

plot_confusion_matrix(all_labels_attention_only, all_preds_attention_only, accuracy_attention_only)

# Save the trained attention layers
torch.save(net_with_attention.state_dict(), 'net_with_attention_trained.pth')

Train Full Network with Attention Layers

In [ ]:

# Enable training for all parameters in the network with attention
for param in net_with_attention.parameters():
    param.requires_grad = True

# Train and evaluate the full network including the attention layers
net_with_attention, lossi_attention_full, accuracies_attention_full, activations_by_epoch_attention_full = train_network(
    net_with_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma)

(accuracy_attention_full, all_preds_attention_full, 
    all_labels_attention_full) = evaluate_network(net_with_attention, test_gen)

plot_confusion_matrix(all_labels_attention_full, all_preds_attention_full, 
    accuracy_attention_full)

# Save the trained attention layers
torch.save(net_with_attention.state_dict(), 'net_with_attention_trained.pth')
# Compute mutual information for the network with attention
# labels_raw = next(iter(train_gen))[1]
# mi_by_epoch_with_attention = compute_mutual_information_by_epoch(activations_by_epoch_attention_full, labels_raw, num_epochs, dendrite_per_soma)

# # Plot mutual information for d1
# plot_mutual_information_by_epoch(mi_by_epoch_with_attention, dendrite_per_soma, 'd1')

# # Plot mutual information for d2
# plot_mutual_information_by_epoch(mi_by_epoch_with_attention, dendrite_per_soma, 'd2')

# # Plot mutual information for sum_mod
# plot_mutual_information_by_epoch(mi_by_epoch_with_attention, dendrite_per_soma, 'sum_mod')



In [ ]:
# Train and evaluate the network without attention
net_without_attention, lossi_without_attention, accuracies_without_attention, activations_by_epoch_without_attention = train_network(
    net_without_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma, capture_activations=True)

# Get a batch of data
data_iter = iter(train_gen)
images, labels = data_iter.next()

# Compute mutual information for the network with attention
mi_by_epoch_with_attention = compute_mutual_information_by_epoch(activations_by_epoch_attention_full, labels, num_epochs, dendrite_per_soma)

# Plot mutual information for d1
plot_mutual_information_by_epoch(mi_by_epoch_with_attention, dendrite_per_soma, 'd1')

Train and Test Network without Attention After Full Training

In [ ]:
# Load the pre-trained weights without attention
net_without_attention.load_state_dict(torch.load('net_without_attention.pth'))

# Train and evaluate the network without attention after full training
net_without_attention, lossi_no_attention_full, accuracies_no_attention_full = train_network(
    net_without_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma)

(accuracy_no_attention_full, all_preds_no_attention_full, 
    all_labels_no_attention_full) = evaluate_network(net_without_attention, test_gen)

plot_confusion_matrix(all_labels_no_attention_full, all_preds_no_attention_full, 
    accuracy_no_attention_full)

Compute and Visualize Metrics for Network without Attention After Full Training

In [ ]:
# # Compute and visualize various metrics for the network without attention after full training
# activations_no_attention_full = compute_information_flow(
#     net_without_attention, transform_images(next(iter(train_gen))[0].view(-1, 28*28)).to(device), 
#     'dendrite_layers')

# mutual_info_no_attention_full = compute_mutual_information(
#     transform_images(next(iter(train_gen))[0].view(-1, 28*28)).cpu().numpy(), 
#     activations_no_attention_full)

# #input_variance_no_attention_full, output_variance_no_attention_full = analyze_variance_signals(
# #    transform_images(next(iter(train_gen))[0].view(-1, 28*28)), net_without_attention.soma_act)
    
# # weights_no_attention_full = analyze_weights(net_without_attention)


# plt.plot(mutual_info_no_attention_full)


# Assuming mutual_info_no_attention_full, mutual_info_attention_full, etc., are computed

# Example usage
num_epochs = 50  # Specify the number of epochs
net_with_attention = NetWithAttention(input_size, inhibitory_units_n, synapses_per_inhibitory_unit, dendrite_layers_n, dendrite_branch_n, inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, shunting, soma_n).to(device)
net_with_attention.initWeights()

# Train network with capturing activations
net_with_attention, lossi_attention, accuracies_attention, activations_by_epoch_attention = train_network(net_with_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma, capture_activations=True)

labels_raw = next(iter(train_gen))[1]
mi_by_epoch_attention = compute_mutual_information_by_epoch(activations_by_epoch_attention, labels_raw, num_epochs, dendrite_per_soma)

# Plot mutual information for d1
plot_mutual_information_by_epoch(mi_by_epoch_attention, dendrite_per_soma, 'd1')

# Plot mutual information for d2
plot_mutual_information_by_epoch(mi_by_epoch_attention, dendrite_per_soma, 'd2')

# Plot mutual information for sum_mod
plot_mutual_information_by_epoch(mi_by_epoch_attention, dendrite_per_soma, 'sum_mod')



In [ ]:
# Assuming the mutual information functions are defined

# Train and evaluate the network without attention
net_without_attention, lossi_without_attention, accuracies_without_attention, activations_by_epoch_without_attention = train_network(
    net_without_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma, capture_activations=True)

# Extract labels from the training data
labels_raw = next(iter(train_gen))[1]
labels_raw = labels_raw if labels_raw.dim() == 2 else labels_raw.unsqueeze(1)

# Compute mutual information for the network without attention
mi_by_epoch_without_attention = compute_mutual_information_by_epoch(activations_by_epoch_without_attention, labels_raw, num_epochs, dendrite_per_soma)

# Plot mutual information for d1
plot_mutual_information_by_epoch(mi_by_epoch_without_attention, dendrite_per_soma, 'd1')

# Plot mutual information for d2
plot_mutual_information_by_epoch(mi_by_epoch_without_attention, dendrite_per_soma, 'd2')

# Plot mutual information for sum_mod
plot_mutual_information_by_epoch(mi_by_epoch_without_attention, dendrite_per_soma, 'sum_mod')

In [ ]:
# Build the models
net_without_attention = Net(input_size, inhibitory_units_n, synapses_per_inhibitory_unit, dendrite_layers_n, dendrite_branch_n, 
                            inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, shunting, soma_n).to(device)
net_without_attention.initWeights()

net_with_attention = NetWithAttention(input_size, inhibitory_units_n, synapses_per_inhibitory_unit, dendrite_layers_n, dendrite_branch_n, 
                                      inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, shunting, soma_n).to(device)
net_with_attention.initWeights()

# Train and evaluate the network without attention
net_without_attention, lossi_without_attention, accuracies_without_attention, activations_by_epoch_without_attention = train_network(
    net_without_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma, capture_activations=True)

accuracy_without_attention, all_preds_without_attention, all_labels_without_attention = evaluate_network(net_without_attention, test_gen)
plot_confusion_matrix(all_labels_without_attention, all_preds_without_attention, accuracy_without_attention)

# Save the pre-trained weights without attention
torch.save(net_without_attention.state_dict(), 'net_without_attention.pth')

# Load pre-trained weights into the network with attention
net_with_attention.load_state_dict(net_without_attention.state_dict(), strict=False)

# Train and evaluate the attention layers
for param in net_with_attention.parameters():
    param.requires_grad = False

for layer in net_with_attention.attention_layers:
    for param in layer.parameters():
        param.requires_grad = True

net_with_attention, lossi_attention_only, accuracies_attention_only, activations_by_epoch_attention_only = train_network(
    net_with_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma, capture_activations=True)

(accuracy_attention_only, all_preds_attention_only, 
    all_labels_attention_only) = evaluate_network(net_with_attention, test_gen)

plot_confusion_matrix(all_labels_attention_only, all_preds_attention_only, 
    accuracy_attention_only)

# Enable training for all parameters in the network with attention
for param in net_with_attention.parameters():
    param.requires_grad = True

# Train and evaluate the full network including the attention layers
net_with_attention, lossi_attention_full, accuracies_attention_full, activations_by_epoch_attention_full = train_network(
    net_with_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma, capture_activations=True)

(accuracy_attention_full, all_preds_attention_full, 
    all_labels_attention_full) = evaluate_network(net_with_attention, test_gen)

plot_confusion_matrix(all_labels_attention_full, all_preds_attention_full, 
    accuracy_attention_full)

# Compute mutual information for the network with attention
labels_raw = next(iter(train_gen))[1]
labels_raw = labels_raw if labels_raw.dim() == 2 else labels_raw.unsqueeze(1)
mi_by_epoch_with_attention = compute_mutual_information_by_epoch(activations_by_epoch_attention_full, labels_raw, num_epochs, dendrite_per_soma)

# Plot mutual information for d1
plot_mutual_information_by_epoch(mi_by_epoch_with_attention, dendrite_per_soma, 'd1')

# Plot mutual information for d2
plot_mutual_information_by_epoch(mi_by_epoch_with_attention, dendrite_per_soma, 'd2')

# Plot mutual information for sum_mod
plot_mutual_information_by_epoch(mi_by_epoch_with_attention, dendrite_per_soma, 'sum_mod')

In [ ]:
# Assuming mutual_info_no_attention_full, mutual_info_attention_full, etc., are computed
labels_raw = next(iter(train_gen))[1]
labels_raw = labels_raw if labels_raw.dim() == 2 else labels_raw.unsqueeze(1)

# Check if the labels_raw has the correct dimensions and values
print(f"labels_raw shape: {labels_raw.shape}")
print(f"labels_raw: {labels_raw}")

# Compute mutual information for the network with attention
mi_by_epoch_with_attention = compute_mutual_information_by_epoch(activations_by_epoch_attention_full, labels_raw, num_epochs, dendrite_per_soma)

# Plot mutual information for d1
plot_mutual_information_by_epoch(mi_by_epoch_with_attention, dendrite_per_soma, 'd1')

Visualizing Dendrite Activations by Label for Network without Attention After Full Training

In [ ]:
def plot_dendrite_activations_by_label(net, train_gen, num_dendrites, num_labels):
    activations_by_label = {i: [] for i in range(num_labels)}

    def hook_fn(module, input, output):
        labels = input[1].cpu().detach().numpy().flatten()
        for i in range(num_labels):
            mask = (labels == i)
            if mask.any():
                activations_by_label[i].append(output[mask, :].cpu().detach().numpy())

    hooks = []
    for layer in net.dendrite_layers:
        hooks.append(layer.register_forward_hook(hook_fn))

    # Perform a forward pass to capture activations
    with torch.no_grad():
        for images_raw, labels_raw in train_gen:
            images = transform_images(images_raw.view(-1, 28*28)).to(device)
            labels = sum_labels(labels_raw).to(device)
            net(images, labels)  # Forward pass with images and labels
            break  # Only need one batch for visualization

    for hook in hooks:
        hook.remove()

    print(num_labels)
    
    # Plot activations
    for i in range(num_labels):
        if len(activations_by_label[i]) == 0:
            continue  # Skip if no activations captured for this label
        activations = np.concatenate(activations_by_label[i], axis=0)
        plt.figure(figsize=(12, 6))
        plt.title(f'Dendrite Activations for Label {i}')
        plt.imshow(activations.T, aspect='auto', cmap='viridis')
        plt.colorbar()
        plt.xlabel('Samples')
        plt.ylabel('Dendrite Activations')
        plt.show()

# Visualize dendrite activations by label for the network without attention after full training
plot_dendrite_activations_by_label(net_without_attention, train_gen, dendrite_per_soma, num_classes)

Training and Testing Network with Attention (Only Training Attention Layers)

In [ ]:
# Freeze all layers except the attention layers
for name, param in net_with_attention.named_parameters():
    if not name.startswith('attention_layers'):
        param.requires_grad = False

# Train and evaluate the attention layers with the pretrained weights from the first network
net_with_attention, lossi_attention_only, accuracies_attention_only = train_network(
    net_with_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma)

accuracy_attention_only, all_preds_attention_only, all_labels_attention_only = evaluate_network(
    net_with_attention, test_gen)
    
plot_confusion_matrix(all_labels_attention_only, all_preds_attention_only, accuracy_attention_only)

Training and Testing Full Network with Attention

In [ ]:
# Unfreeze all layers for full training
for param in net_with_attention.parameters():
    param.requires_grad = True

# Train and evaluate the full network including the attention layers
net_with_attention_full, lossi_attention_full, accuracies_attention_full = train_network(
    net_with_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma)

(accuracy_attention_full, all_preds_attention_full,
    all_labels_attention_full) = evaluate_network(net_with_attention_full, test_gen)
    
plot_confusion_matrix(all_labels_attention_full, all_preds_attention_full, accuracy_attention_full)

Analyzing and Visualizing Attention Weights

In [ ]:
def visualize_attention_weights(net, layer_name):
    attention_weights = []

    def hook_fn(module, input, output):
        attention_weights.append(module.weight.data.detach().cpu().numpy())

    hooks = []
    for layer in getattr(net, layer_name):
        hooks.append(layer.register_forward_hook(hook_fn))

    # Perform a forward pass to capture attention weights
    with torch.no_grad():
        for images_raw, _ in train_gen:
            images = transform_images(images_raw.view(-1, 28*28)).to(device)
            net(images)
            break  # Only need one batch for visualization

    for hook in hooks:
        hook.remove()

    if len(attention_weights) == 0:
        raise ValueError("No attention weights were captured. Ensure the layer names are correct and the model's forward pass is executed properly.")

    # Plot each attention weight separately
    for i, weights in enumerate(attention_weights):
        plt.figure(figsize=(12, 8))
        plt.imshow(weights, aspect='auto', cmap='viridis')
        plt.colorbar()
        plt.title(f'Attention Weights from {layer_name} Layer {i+1}')
        plt.xlabel('Attention Heads')
        plt.ylabel('Weights')
        plt.show()
    
    return attention_weights


def plot_training_metrics(losses, accuracies):
    # Plot Losses
    plt.figure(figsize=(10, 5))
    plt.plot(losses)
    plt.title("Loss over Time")
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.show()

    # Plot Accuracies
    plt.figure(figsize=(10, 5))
    plt.plot(accuracies)
    plt.title("Training Accuracy over Time")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.show()
    
# Visualize attention weights after full training
attention_weights_full = visualize_attention_weights(net_with_attention_full, 'attention_layers')

# Compute and visualize various metrics for the full network with attention after full training
activations_attention_full = compute_information_flow(
    net_with_attention_full, transform_images(next(iter(train_gen))[0].view(-1, 28*28)).to(device), 
    'dendrite_layers')

mutual_info_attention_full = compute_mutual_information(
    transform_images(next(iter(train_gen))[0].view(-1, 28*28)).cpu().numpy(), 
    activations_attention_full)

input_variance_attention_full, output_variance_attention_full = analyze_variance_signals(
    transform_images(next(iter(train_gen))[0].view(-1, 28*28)), net_with_attention_full.soma_act)
    
weights_attention_full = analyze_weights(net_with_attention_full)

# Plot training metrics for the full network with attention
plot_training_metrics(lossi_attention_full, accuracies_attention_full)

Removing Attention Layers for Testing and Analysis

In [ ]:


# Initialize the network without attention layers using the fully trained weights
net_without_attention_from_full = NetWithoutAttention(net_with_attention_full).to(device)

# Evaluate the network without attention layers after full training
(accuracy_without_attention_from_full,
 all_preds_without_attention_from_full,
 all_labels_without_attention_from_full) = evaluate_network(net_without_attention_from_full, test_gen)

plot_confusion_matrix(all_labels_without_attention_from_full,
    all_preds_without_attention_from_full, accuracy_without_attention_from_full)

# Compute and visualize various metrics for the network without attention layers after full training
activations_without_attention_from_full = compute_information_flow(
    net_without_attention_from_full, transform_images(next(iter(train_gen))[0].view(-1, 28*28)).to(device),
    'dendrite_layers')

mutual_info_without_attention_from_full = compute_mutual_information(
    transform_images(next(iter(train_gen))[0].view(-1, 28*28)).cpu().numpy(),
    activations_without_attention_from_full)

input_variance_without_attention_from_full, output_variance_without_attention_from_full = analyze_variance_signals(
    transform_images(next(iter(train_gen))[0].view(-1, 28*28)), net_without_attention_from_full.soma_act)

weights_without_attention_from_full = analyze_weights(net_without_attention_from_full)

Plotting Dendrite Activations by Label

In [ ]:
# def plot_dendrite_activations_by_label1(net, train_gen, num_dendrites, num_labels):
#     activations_by_label = {i: [] for i in range(num_labels)}

#     def hook_fn(module, input, output):
#         labels = input[1].reshape(-1)
#         for i in range(num_labels):
#             mask = (labels == i).reshape(-1)
#             activations_by_label[i].append(output[mask].cpu().detach().numpy())

#     hooks = []
#     for layer in net.dendrite_layers:
#         hooks.append(layer.register_forward_hook(hook_fn))

#     with torch.no_grad():
#         for images_raw, labels_raw in train_gen:
#             images = transform_images(images_raw.view(-1, 28*28)).to(device)
#             labels = sum_labels(labels_raw).to(device)
#             net(images)  # Forward pass with only images
#             break  # Only need one batch for visualization

#     for hook in hooks:
#         hook.remove()

#     plt.figure(figsize=(12, 8))
#     for i in range(num_labels):
#         if activations_by_label[i]:  # Check if the list is not empty
#             activations_concat = np.concatenate(activations_by_label[i], axis=0)  # Concatenate all activations for the label
#             activations_mean = np.mean(activations_concat, axis=0)  # Compute the mean activation
#             for dendrite in range(num_dendrites):
#                 plt.plot(activations_mean[dendrite], label=f'Dendrite {dendrite}')
#             plt.title(f'Dendrite Activations for Label {i}')
#             plt.xlabel('Neuron Index')
#             plt.ylabel('Activation')
#             plt.legend()
#             plt.show()

# Example usage
num_labels = 10
plot_dendrite_activations_by_label(net_without_attention_from_full, train_gen, dendrite_per_soma, num_labels)


# def plot_dendrite_activations_by_label(net, train_gen, num_dendrites, num_labels):
#     activations_by_label = [[] for _ in range(num_labels)]

#     def hook_fn(module, input, output):
#         labels = input[1].cpu().numpy().reshape(-1)
#         for i in range(num_labels):
#             mask = (labels == i)
#             if mask.sum() > 0:
#                 activations_by_label[i].append(output[mask, :].cpu().detach().numpy())

#     hooks = []
#     for layer in net.dendrite_layers:
#         hooks.append(layer.register_forward_hook(hook_fn))

#     # Perform a forward pass to capture activations
#     with torch.no_grad():
#         for images_raw, labels_raw in train_gen:
#             images = transform_images(images_raw.view(-1, 28*28)).to(device)
#             labels = sum_labels(labels_raw).to(device)
#             net(images)
#             break  # Only need one batch for visualization

#     for hook in hooks:
#         hook.remove()

#     for i in range(num_labels):
#         plt.figure(figsize=(12, 8))
#         for dendrite in range(num_dendrites):
#             plt.plot(np.mean(activations_by_label[i], axis=0)[:, dendrite], label=f'Dendrite {dendrite}')
#         plt.title(f'Dendrite Activations for Label {i}')
#         plt.xlabel('Neuron Index')
#         plt.ylabel('Activation')
#         plt.legend()
#         plt.show()

# # Example usage
# plot_dendrite_activations_by_label(net_without_attention_from_full, train_gen, dendrite_per_soma, num_labels)

# Example usage


Visualizing What Each Dendrite Weight Represents About the Input

In [ ]:
def plot_weight_representations(net, num_labels):
    weight_representations = [[] for _ in range(num_labels)]

    def hook_fn(module, input, output):
        weights = module.weight.data.cpu().detach().numpy()
        labels = input[1].cpu().numpy().reshape(-1)
        for i in range(num_labels):
            mask = (labels == i)
            if mask.sum() > 0:
                weight_representations[i].append(weights[mask].mean(axis=0))

    hooks = []
    for layer in net.dendrite_layers:
        hooks.append(layer.register_forward_hook(hook_fn))

    # Perform a forward pass to capture weight representations
    with torch.no_grad():
        for images_raw, labels_raw in train_gen:
            images = transform_images(images_raw.view(-1, 28*28)).to(device)
            labels = sum_labels(labels_raw).to(device)
            net(images)
            break  # Only need one batch for visualization

    for hook in hooks:
        hook.remove()

    for i in range(num_labels):
        plt.figure(figsize=(12, 8))
        plt.imshow(np.array(weight_representations[i]).T, aspect='auto', cmap='viridis')
        plt.colorbar()
        plt.title(f'Weight Representations for Label {i}')
        plt.xlabel('Neuron Index')
        plt.ylabel('Dendrite Index')
        plt.show()

# Example usage
plot_weight_representations(net_without_attention_from_full, num_labels)

# Example usage
plot_weight_representations(net_without_attention_from_full, num_labels)



Analyzing the Inhibition Effect

In [ ]:
def analyze_inhibition_effect(net, num_labels):
    inhibition_effect = [[] for _ in range(num_labels)]

    def hook_fn(module, input, output):
        inhib_act = module.inhibition_act.cpu().detach().numpy()
        for i in range(num_labels):
            inhibition_effect[i].append(inhib_act[input[1] == i, :])

    hooks = []
    hooks.append(net.inputs_to_inhibition.register_forward_hook(hook_fn))

    # Perform a forward pass to capture inhibition activations
    with torch.no_grad():
        for images_raw, labels_raw in train_gen:
            images = transform_images(images_raw.view(-1, 28*28)).to(device)
            labels = sum_labels(labels_raw).to(device)
            net(images)  # Forward pass with only images
            break  # Only need one batch for visualization

    for hook in hooks:
        hook.remove()

    # Concatenate inhibition activations and plot
    for label in range(num_labels):
        inhibition_effect[label] = np.concatenate(inhibition_effect[label], axis=0)
        plt.figure(figsize=(12, 6))
        plt.imshow(inhibition_effect[label], aspect='auto', cmap='viridis')
        plt.colorbar()
        plt.title(f'Inhibition Effect for Label {label}')
        plt.xlabel('Inhibition Unit Index')
        plt.ylabel('Sample Index')
        plt.show()

# Example usage
analyze_inhibition_effect(net_without_attention_from_full, num_labels)

Comparing Attention and No-Attention Models

In [ ]:
def compare_models(net_no_attention, net_attention, test_gen):
    accuracy_no_attention, _, _ = evaluate_network(net_no_attention, test_gen)
    accuracy_attention, _, _ = evaluate_network(net_attention, test_gen)
    
    print(f"Accuracy without attention: {accuracy_no_attention:.3f}%")
    print(f"Accuracy with attention: {accuracy_attention:.3f}%")
    
    if accuracy_attention > accuracy_no_attention:
        print("The model with attention performs better.")
    elif accuracy_attention < accuracy_no_attention:
        print("The model without attention performs better.")
    else:
        print("Both models perform equally well.")

# Example usage
compare_models(net_without_attention_from_full, net_with_attention_full_train, test_gen)

Analyzing Weights and Activations

In [ ]:
def analyze_model(net, train_gen, num_labels, model_name):
    # Compute and visualize various metrics
    activations = compute_information_flow(net, transform_images(next(iter(train_gen))[0].view(-1, 28*28)).to(device), 'dendrite_layers')
    mutual_info = compute_mutual_information(transform_images(next(iter(train_gen))[0].view(-1, 28*28)).cpu().numpy(), activations)
    input_variance, output_variance = analyze_variance_signals(transform_images(next(iter(train_gen))[0].view(-1, 28*28)), net.soma_act)
    weights = analyze_weights(net)

    print(f"Model: {model_name}")
    print("Mutual Information:", mutual_info)
    print("Input Variance:", input_variance)
    print("Output Variance:", output_variance)
    plot_dendrite_activations_by_label(net, train_gen, dendrite_per_soma, num_labels)

# Example usage
analyze_model(net_without_attention, train_gen, num_labels, "Without Attention")
analyze_model(net_with_attention_full_train, train_gen, num_labels, "With Attention")

Visualizing Attention Weights and Evaluating Attention Model

In [ ]:
# Visualize attention weights
attention_weights = visualize_attention_weights(net_with_attention_full_train, 'attention_layers')

# Evaluate the network with attention layers removed
net_with_attention_full_train.eval()
for layer in net_with_attention_full_train.attention_layers:
    for param in layer.parameters():
        param.requires_grad = False

accuracy_attention_removed, all_preds_attention_removed, all_labels_attention_removed = evaluate_network(net_with_attention_full_train, test_gen)
plot_confusion_matrix(all_labels_attention_removed, all_preds_attention_removed, accuracy_attention_removed)

# Analyze the model with attention layers removed
analyze_model(net_with_attention_full_train, train_gen, num_labels, "With Attention Layers Removed")

Comparing Metrics Before and After Removing Attention Layers

In [ ]:
# Compare metrics before and after removing attention layers
activations_with_attention_removed = compute_information_flow(net_with_attention_full_train, transform_images(next(iter(train_gen))[0].view(-1, 28*28)).to(device), 'dendrite_layers')
mutual_info_with_attention_removed = compute_mutual_information(transform_images(next(iter(train_gen))[0].view(-1, 28*28)).cpu().numpy(), activations_with_attention_removed)
input_variance_with_attention_removed, output_variance_with_attention_removed = analyze_variance_signals(transform_images(next(iter(train_gen))[0].view(-1, 28*28)), net_with_attention_full_train.soma_act)
weights_with_attention_removed = analyze_weights(net_with_attention_full_train)

# Plot and compare the mutual information
plt.figure(figsize=(10, 6))
plt.plot(mutual_info_attention, label='With Attention')
plt.plot(mutual_info_with_attention_removed, label='Without Attention')
plt.title('Mutual Information Comparison')
plt.xlabel('Neuron Index')
plt.ylabel('Mutual Information')
plt.legend()
plt.show()

# Plot and compare input and output variances
plt.figure(figsize=(10, 6))
plt.subplot(1, 2, 1)
plt.plot(input_variance_attention, label='With Attention')
plt.plot(input_variance_with_attention_removed, label='Without Attention')
plt.title('Input Variance Comparison')
plt.xlabel('Neuron Index')
plt.ylabel('Variance')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(output_variance_attention, label='With Attention')
plt.plot(output_variance_with_attention_removed, label='Without Attention')
plt.title('Output Variance Comparison')
plt.xlabel('Neuron Index')
plt.ylabel('Variance')
plt.legend()

plt.show()

Quantifying Information in Dendrite Weights

In [ ]:
# Function to quantify information in dendrite weights
def quantify_information_in_weights(weights, bins=30):
    from sklearn.metrics import mutual_info_score
    
    weight_info = {}
    for name, weight in weights.items():
        weight_flat = weight.flatten()
        # Discretize weights into bins
        weight_bins = np.digitize(weight_flat, bins=np.histogram(weight_flat, bins=bins)[1])
        mutual_info = mutual_info_score(weight_bins, weight_flat)
        weight_info[name] = mutual_info
    
    return weight_info

# Quantify information in dendrite weights before and after adding attention
weight_info_no_attention = quantify_information_in_weights(weights_no_attention)
weight_info_with_attention = quantify_information_in_weights(weights_attention)
weight_info_with_attention_removed = quantify_information_in_weights(weights_with_attention_removed)

# Plotting the information in weights
plt.figure(figsize=(12, 8))
names = list(weight_info_no_attention.keys())
info_no_attention = list(weight_info_no_attention.values())
info_with_attention = list(weight_info_with_attention.values())
info_with_attention_removed = list(weight_info_with_attention_removed.values())

x = np.arange(len(names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 8))
rects1 = ax.bar(x - width, info_no_attention, width, label='No Attention')
rects2 = ax.bar(x, info_with_attention, width, label='With Attention')
rects3 = ax.bar(x + width, info_with_attention_removed, width, label='With Attention Removed')

ax.set_xlabel('Layer')
ax.set_ylabel('Mutual Information')
ax.set_title('Information in Dendrite Weights by Layer')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha="right")
ax.legend()

plt.tight_layout()
plt.show()

Visualizing Attention Mechanisms

In [ ]:
# Function to visualize the attention mechanism
def visualize_attention_mechanism(net, layer_name, train_gen):
    attention_activations = []

    def hook_fn(module, input, output):
        attention_activations.append(output.cpu().detach().numpy())

    hooks = []
    for layer in getattr(net, layer_name):
        hooks.append(layer.register_forward_hook(hook_fn))

    with torch.no_grad():
        for images_raw, _ in train_gen:
            images = transform_images(images_raw.view(-1, 28*28)).to(device)
            net(images)
            break

    for hook in hooks:
        hook.remove()

    attention_activations = np.concatenate(attention_activations, axis=0)

    return attention_activations

# Visualize attention mechanisms
attention_activations = visualize_attention_mechanism(net_with_attention, 'attention_layers', train_gen)

# Plotting the attention activations
plt.figure(figsize=(12, 8))
plt.imshow(attention_activations, aspect='auto', cmap='viridis')
plt.colorbar()
plt.title('Attention Activations')
plt.xlabel('Attention Heads')
plt.ylabel('Activation')
plt.show()

Evaluating Network Performance Without Attention Layers After Full Training

In [ ]:
# Evaluate the network without attention layers after full training
accuracy_no_attention, all_preds_no_attention, all_labels_no_attention = evaluate_network(net_without_attention, test_gen)
plot_confusion_matrix(all_labels_no_attention, all_preds_no_attention, accuracy_no_attention)

# Compute and visualize various metrics for the network without attention after full training
activations_no_attention = compute_information_flow(net_without_attention, transform_images(next(iter(train_gen))[0].view(-1, 28*28)).to(device), 'dendrite_layers')
mutual_info_no_attention = compute_mutual_information(transform_images(next(iter(train_gen))[0].view(-1, 28*28)).cpu().numpy(), activations_no_attention)
input_variance_no_attention, output_variance_no_attention = analyze_variance_signals(transform_images(next(iter(train_gen))[0].view(-1, 28*28)), net_without_attention.soma_act)
weights_no_attention = analyze_weights(net_without_attention)

Train and Evaluate Network With Pretrained Layers and Attention Layers

In [ ]:
# Step 2: Add attention layers but use the pre-trained layers from Step 1 and only train the attention layers

# Freeze all layers except attention layers
for param in net_with_attention.parameters():
    param.requires_grad = False
for layer in net_with_attention.attention_layers:
    for param in layer.parameters():
        param.requires_grad = True

# Train and evaluate the network with attention layers, using pre-trained weights from Step 1
net_with_attention, lossi_attention_pretrained, accuracies_attention_pretrained = train_network(net_with_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma)
accuracy_attention_pretrained, all_preds_attention_pretrained, all_labels_attention_pretrained = evaluate_network(net_with_attention, test_gen)
plot_confusion_matrix(all_labels_attention_pretrained, all_preds_attention_pretrained, accuracy_attention_pretrained)

# Visualize attention weights
attention_weights_pretrained = visualize_attention_weights(net_with_attention, 'attention_layers')

# Compute and visualize various metrics with attention using pre-trained weights
activations_attention_pretrained = compute_information_flow(net_with_attention, transform_images(next(iter(train_gen))[0].view(-1, 28*28)).to(device), 'dendrite_layers')
mutual_info_attention_pretrained = compute_mutual_information(transform_images(next(iter(train_gen))[0].view(-1, 28*28)).cpu().numpy(), activations_attention_pretrained)
input_variance_attention_pretrained, output_variance_attention_pretrained = analyze_variance_signals(transform_images(next(iter(train_gen))[0].view(-1, 28*28)), net_with_attention.soma_act)
weights_attention_pretrained = analyze_weights(net_with_attention)

Train and Evaluate Network Including All Layers (Step 3)

In [ ]:
# Step 3: Train and test a full network including all layers (retrain all layers with attention)

# Unfreeze all layers for full training
for param in net_with_attention.parameters():
    param.requires_grad = True

# Train and evaluate the full network with attention layers
net_with_attention_full, lossi_attention_full, accuracies_attention_full = train_network(net_with_attention, train_gen, test_gen, num_epochs, lr, scheduler_gamma)
accuracy_attention_full, all_preds_attention_full, all_labels_attention_full = evaluate_network(net_with_attention_full, test_gen)
plot_confusion_matrix(all_labels_attention_full, all_preds_attention_full, accuracy_attention_full)

# Visualize attention weights for the fully trained network
attention_weights_full = visualize_attention_weights(net_with_attention_full, 'attention_layers')

# Compute and visualize various metrics with attention after full training
activations_attention_full = compute_information_flow(net_with_attention_full, transform_images(next(iter(train_gen))[0].view(-1, 28*28)).to(device), 'dendrite_layers')
mutual_info_attention_full = compute_mutual_information(transform_images(next(iter(train_gen))[0].view(-1, 28*28)).cpu().numpy(), activations_attention_full)
input_variance_attention_full, output_variance_attention_full = analyze_variance_signals(transform_images(next(iter(train_gen))[0].view(-1, 28*28)), net_with_attention_full.soma_act)
weights_attention_full = analyze_weights(net_with_attention_full)

Remove Attention Layers and Evaluate Network (Step 4)

In [ ]:
# Step 4: Remove attention layers and test the performance and quantifications of other layers

# Create a new network without attention layers but with the same weights as the fully trained network
net_without_attention_full = Net(input_size, inhibitory_units_n, synapses_per_inhibitory_unit, dendrite_layers_n, dendrite_branch_n,
                                 inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, shunting, soma_n).to(device)

# Copy weights from the fully trained network with attention layers
net_without_attention_full.load_state_dict(net_with_attention_full.state_dict(), strict=False)

# Evaluate the network without attention layers
accuracy_no_attention_full, all_preds_no_attention_full, all_labels_no_attention_full = evaluate_network(net_without_attention_full, test_gen)
plot_confusion_matrix(all_labels_no_attention_full, all_preds_no_attention_full, accuracy_no_attention_full)

# Compute and visualize various metrics for the network without attention after full training
activations_no_attention_full = compute_information_flow(net_without_attention_full, transform_images(next(iter(train_gen))[0].view(-1, 28*28)).to(device), 'dendrite_layers')
mutual_info_no_attention_full = compute_mutual_information(transform_images(next(iter(train_gen))[0].view(-1, 28*28)).cpu().numpy(), activations_no_attention_full)
input_variance_no_attention_full, output_variance_no_attention_full = analyze_variance_signals(transform_images(next(iter(train_gen))[0].view(-1, 28*28)), net_without_attention_full.soma_act)
weights_no_attention_full = analyze_weights(net_without_attention_full)

Visualize Activations by Label (Step 4)

In [ ]:
# Visualize dendrite activations by label for the network without attention after full training

plot_dendrite_activations_by_label(net_without_attention_full, train_gen, dendrite_per_soma, num_labels)

Summary and Conclusion

In [ ]:
# Summary and Conclusion
print("Summary and Conclusion:")
print("1. We trained a network without attention layers.")
print("2. We added attention layers and trained only the attention layers.")
print("3. We trained a full network including the attention layers.")
print("4. We removed attention layers and tested the performance of the network without attention after full training.")
print("5. We visualized what each dendrite weight and dendrite represents about the input during the training.")
print("6. We analyzed the dendrite activations by label for different scenarios.")
print("7. The results indicate that the network's performance and activations can be effectively analyzed and visualized for further insights.")

# Save the trained models for future use
torch.save(net.state_dict(), 'net_without_attention.pt')
torch.save(net_with_attention.state_dict(), 'net_with_attention.pt')

In [ ]:
from graphviz import Digraph

# Create a new directed graph
dot = Digraph(comment='Basic Network without Attention')

# Input Layer
dot.node('I', 'Input Layer\n(MNIST Digits)')

# Inhibition Layer
dot.node('In', 'Inhibition Layer\n(TopKLinear)')

# Excitatory Inputs
dot.node('Ex', 'Excitatory Inputs\n(Synapses)')

# Dendrite Layers
dot.node('D1', 'Dendrite Layer 1\n(DendriteBranchLayer\n(BlockLinear))')
dot.node('D2', 'Dendrite Layer 2\n(DendriteBranchLayer\n(BlockLinear))')

# Soma Layer
dot.node('S', 'Soma Layer\n(BlockLinear)')

# Connecting the nodes
dot.edges([('I', 'In'), ('I', 'Ex'), ('In', 'D1'), ('Ex', 'D1'), ('D1', 'D2'), ('D2', 'S'), ('In', 'D2'), ('Ex', 'D2')])


dot.render('/Users/houmansafaai/Library/CloudStorage/OneDrive-HarvardUniversity/NeuroAI/dendritic-modeling-notebook/Results/Basic_Network_without_Attention', format='png', view=True)

# Create a new directed graph
dot = Digraph(comment='Network with Attention')

# Input Layer
dot.node('I', 'Input Layer\n(MNIST Digits)')

# Inhibition Layer
dot.node('In', 'Inhibition Layer\n(TopKLinear)')

# Excitatory Inputs
dot.node('Ex', 'Excitatory Inputs\n(Synapses)')

# Dendrite Layers
dot.node('D1', 'Dendrite Layer 1\n(DendriteBranchLayer\n(BlockLinear))')
dot.node('D2', 'Dendrite Layer 2\n(DendriteBranchLayer\n(BlockLinear))')

# Attention Layers
dot.node('A1', 'Attention Layer 1\n(Linear)')
dot.node('A2', 'Attention Layer 2\n(Linear)')

# Soma Layer
dot.node('S', 'Soma Layer\n(BlockLinear)')

# Connecting the nodes
dot.edges([('I', 'In'), ('I', 'Ex'), ('In', 'D1'), ('Ex', 'D1'), ('D1', 'A1'), ('A1', 'D2'), ('D2', 'A2'), ('A2', 'S'), ('In', 'D2'), ('Ex', 'D2')])



dot.render('/Users/houmansafaai/Library/CloudStorage/OneDrive-HarvardUniversity/NeuroAI/dendritic-modeling-notebook/Results/Basic_Network_with_Attention', format='png', view=True)

# Continue with the rest of the blocks for the network with attention...

In [ ]:
# Check if BlockLinear is used in any other part of the network
from torch import nn

for name, module in net_without_attention_full.named_modules():
    if isinstance(module, BlockLinear):
        print(f"BlockLinear is used in layer: {name}")

Analyze Initial Weights and Activations

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Function to visualize dendrite activations
def visualize_dendrite_activations(net, inputs, epoch, dendrite_per_soma, soma_n):
    net.eval()
    with torch.no_grad():
        activations = net(inputs)
    activations = activations.cpu().numpy()
    
    plt.figure(figsize=(15, 10))
    for soma in range(soma_n):
        plt.subplot(3, 3, soma + 1)
        for dendrite in range(dendrite_per_soma):
            plt.plot(activations[:, soma * dendrite_per_soma + dendrite], label=f'Dendrite {dendrite}')
        plt.title(f'Soma {soma}')
        plt.xlabel('Sample Index')
        plt.ylabel('Activation')
        plt.legend()
    plt.suptitle(f'Dendrite Activations at Epoch {epoch}')
    plt.show()

# Initialize the network
net = Net(input_size, inhibitory_units_n, synapses_per_inhibitory_unit, dendrite_layers_n, dendrite_branch_n, 
          inhibitory_synapses_per_dendrite_branch, excitatory_synapses_per_dendrite_branch, shunting, soma_n).to(device)
net.initWeights()

# Visualize initial dendrite activations
inputs = transform_images(next(iter(train_gen))[0].view(-1, 28*28)).to(device)
visualize_dendrite_activations(net, inputs, epoch=0, dendrite_per_soma=dendrite_per_soma, soma_n=soma_n)

Analyze Intermediate Weights and Activations

In [ ]:
# Function to train the network and save intermediate states
def train_network_with_checkpoints(net, train_gen, num_epochs, lr, scheduler_gamma, checkpoints):
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=scheduler_gamma)
    loss_function = torch.nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        net.train()
        for images_raw, labels_raw in train_gen:
            images = transform_images(images_raw.view(-1, 28*28)).to(device)
            labels = sum_labels(labels_raw).to(device)
            
            optimizer.zero_grad()
            outputs = net(images)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()

        scheduler.step()

        if epoch in checkpoints:
            torch.save(net.state_dict(), f'net_checkpoint_epoch_{epoch}.pth')
            visualize_dendrite_activations(net, images, epoch=epoch, dendrite_per_soma=dendrite_per_soma, soma_n=soma_n)

# Train the network with checkpoints at epochs 1, 10, 50, and 100
checkpoints = [1, 10, 50, 100]
train_network_with_checkpoints(net, train_gen, num_epochs=100, lr=0.001, scheduler_gamma=0.95, checkpoints=checkpoints)

Analyze Final Weights and Activations

In [ ]:
# Load the final model and visualize final dendrite activations
net.load_state_dict(torch.load('net_checkpoint_epoch_100.pth'))
visualize_dendrite_activations(net, inputs, epoch=100, dendrite_per_soma=dendrite_per_soma, soma_n=soma_n)

In [ ]:
# Function to visualize attention weights
def visualize_attention_weights(net, epoch):
    attention_weights = net_with_attention.attention_layer.weights.data.cpu().numpy()
    plt.figure(figsize=(10, 5))
    plt.imshow(attention_weights, aspect='auto', cmap='viridis')
    plt.colorbar()
    plt.title(f'Attention Weights at Epoch {epoch}')
    plt.xlabel('Dendrite/Excitatory/Inhibitory Inputs')
    plt.ylabel('Attention Units')
    plt.show()

# Visualize attention weights at different epochs
for epoch in checkpoints:
    net.load_state_dict(torch.load(f'net_checkpoint_epoch_{epoch}.pth'))
    visualize_attention_weights(net, epoch)